In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:

# ==========================================
# S9.5 - CELL 1: ENVIRONMENT SETUP
# ==========================================

from pathlib import Path
import hashlib
import zipfile
import json
import platform

import numpy as np
import pandas as pd

from IPython.display import display

SEED = 42

INPUT_ROOT = Path("/kaggle/input")

OUT = Path("/kaggle/working/S9_5")
OUT.mkdir(parents=True, exist_ok=True)

PRIMARY_METRIC = "AP"

EXPECTED_SOURCE_FILES = 34
EXPECTED_CSV_FILES = 29

EXPECTED_REGISTRY_HASH = (
    "6056188ba850e34c9d6c81d5ea39dbcfabd154b782a15abc2aa43dc3480284a4"
)

EXPECTED_ZIP_HASH = (
    "da99c86e5bd1756f14c2fa6297733ee0aa79b4597ce8f1a987123f2d477c32e3"
)


def sha256_file(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()


print("=== S9.5 ENVIRONMENT ===")

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

print("\nSeed:", SEED)
print("Primary metric:", PRIMARY_METRIC)
print("Scope: Existing CV and validation results")

print("\nInput root:", INPUT_ROOT)
print("Output:", OUT)

print("\n=== SAFETY PROTOCOL ===")

print("Model training: NO")
print("Raw BRFSS required: NO")
print("Test metrics aggregation: NO")
print("FINAL evaluation: NO")

assert INPUT_ROOT.is_dir()
assert OUT.is_dir()

print("\nS9.5 CELL 1: PASS")


=== S9.5 ENVIRONMENT ===
Python: 3.13.18
NumPy: 2.1.3
Pandas: 2.3.3

Seed: 42
Primary metric: AP
Scope: Existing CV and validation results

Input root: /kaggle/input
Output: /kaggle/working/S9_5

=== SAFETY PROTOCOL ===
Model training: NO
Raw BRFSS required: NO
Test metrics aggregation: NO
FINAL evaluation: NO

S9.5 CELL 1: PASS


In [2]:

# ==========================================
# S9.5 - CELL 2: SOURCE INTEGRITY AUDIT
# ==========================================

print("=== LOCATE INPUT DATA ===")

registry_paths = list(
    INPUT_ROOT.rglob("source_registry.csv")
)

zip_paths = list(
    INPUT_ROOT.rglob("S9_5_Aggregation_Inputs.zip")
)

# 1. Locate or extract package

if len(registry_paths) == 1:

    REGISTRY_PATH = registry_paths[0]
    SOURCE_ROOT = REGISTRY_PATH.parent

    print("Input mode: Extracted directory")

elif len(registry_paths) == 0:

    assert len(zip_paths) == 1, zip_paths

    source_zip = zip_paths[0]

    assert sha256_file(source_zip) == EXPECTED_ZIP_HASH

    extract_dir = OUT / "input_snapshot"
    extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(source_zip) as z:

        assert z.testzip() is None

        for name in z.namelist():

            destination = (extract_dir / name).resolve()

            assert destination.is_relative_to(
                extract_dir.resolve()
            )

        z.extractall(extract_dir)

    SOURCE_ROOT = extract_dir / "S9_5_inputs"
    REGISTRY_PATH = SOURCE_ROOT / "source_registry.csv"

    print("Input mode: ZIP archive")
    print("[PASS] Input ZIP SHA-256")

else:

    raise ValueError(
        f"Multiple source registries found: {registry_paths}"
    )

print("Source root:", SOURCE_ROOT)

# 2. Verify registry

assert REGISTRY_PATH.is_file()

assert sha256_file(REGISTRY_PATH) == EXPECTED_REGISTRY_HASH

registry = pd.read_csv(
    REGISTRY_PATH,
    dtype={"source_id": str}
)

required_columns = {
    "source_id",
    "owner",
    "archive",
    "archive_sha256",
    "archive_member",
    "relative_path",
    "category",
    "sha256",
    "bytes",
    "data_rows"
}

assert required_columns.issubset(registry.columns)

assert len(registry) == EXPECTED_SOURCE_FILES
assert registry["source_id"].is_unique
assert registry["relative_path"].is_unique
assert set(registry["owner"]) == {"B", "C"}

print("[PASS] Source registry")

# 3. Verify source files

audit_rows = []

for row in registry.itertuples(index=False):

    path = (SOURCE_ROOT / row.relative_path).resolve()

    assert path.is_relative_to(SOURCE_ROOT.resolve())
    assert path.is_file(), path

    actual_hash = sha256_file(path)

    assert actual_hash == row.sha256, (
        f"Hash mismatch: {row.relative_path}"
    )

    assert path.stat().st_size == row.bytes

    actual_rows = None

    if path.suffix == ".csv":

        df = pd.read_csv(path)
        actual_rows = len(df)

        assert actual_rows == int(row.data_rows)

    audit_rows.append({
        "source_id": row.source_id,
        "owner": row.owner,
        "relative_path": row.relative_path,
        "category": row.category,
        "sha256_match": True,
        "size_match": True,
        "data_rows": actual_rows,
        "status": "PASS"
    })

    print(
        f"[PASS] {row.source_id} | "
        f"{row.owner} | {path.name}"
    )

# 4. Consolidated audit

source_audit = pd.DataFrame(audit_rows)

assert len(source_audit) == 34
assert (source_audit["status"] == "PASS").all()

csv_count = int(
    registry["relative_path"].str.endswith(".csv").sum()
)

assert csv_count == EXPECTED_CSV_FILES

total_csv_rows = int(registry["data_rows"].sum())

assert total_csv_rows == 230

# 5. Save results

source_audit.to_csv(
    OUT / "source_integrity_audit.csv",
    index=False
)

registry.to_csv(
    OUT / "source_registry.csv",
    index=False
)

# 6. Display inventory

print("\n=== SOURCE INVENTORY BY OWNER ===")

display(
    registry.groupby("owner")
    .size()
    .rename("n_files")
    .reset_index()
)

print("\n=== SOURCE INVENTORY BY CATEGORY ===")

display(
    registry.groupby("category")
    .size()
    .rename("n_files")
    .reset_index()
)

print("\n=== SOURCE AUDIT SUMMARY ===")

print("Source files:", len(registry))
print("CSV files:", csv_count)
print("Total CSV rows:", total_csv_rows)
print("Verified:", len(source_audit), "/ 34")
print("Output:", OUT)

print("\nS9.5 CELL 2: PASS")


=== LOCATE INPUT DATA ===
Input mode: Extracted directory
Source root: /kaggle/input/datasets/dinhgiaphong/ai-midterm-s9-aggregation-inputs/S9_5_inputs
[PASS] Source registry
[PASS] SRC001 | B | hyperparams_search.csv
[PASS] SRC002 | B | grid_model_x_resample.csv
[PASS] SRC003 | B | chosen_strategy.csv
[PASS] SRC004 | B | val_metrics.csv
[PASS] SRC005 | B | group_split_sensitivity.csv
[PASS] SRC006 | B | b4_feature_selection_cv.csv
[PASS] SRC007 | B | b4_selected_config.csv
[PASS] SRC008 | B | b4_val_metrics.csv
[PASS] SRC009 | B | b4_selected_features.csv
[PASS] SRC010 | B | b4_k15_features.csv
[PASS] SRC011 | B | b4_permutation_importance.csv
[PASS] SRC012 | B | locked_config.json
[PASS] SRC013 | B | b4_locked_config.json
[PASS] SRC014 | C | results_C_baseline_none.csv
[PASS] SRC015 | C | results_C_appendix_decision_tree.csv
[PASS] SRC016 | C | results_C_baseline_none.csv
[PASS] SRC017 | C | tune_C_logistic.csv
[PASS] SRC018 | C | tune_C_random_forest.csv
[PASS] SRC019 | C | tune_C_d

,owner,n_files
0,B,13
1,C,21



=== SOURCE INVENTORY BY CATEGORY ===


,category,n_files
0,context,5
1,cv_imbalance,3
2,cv_tuning,3
3,cv_tuning_xie,5
4,feature_cv,2
5,feature_metadata,3
6,feature_validation,1
7,group_cv,6
8,threshold_sensitivity,1
9,validation,3



=== SOURCE AUDIT SUMMARY ===
Source files: 34
CSV files: 29
Total CSV rows: 230
Verified: 34 / 34
Output: /kaggle/working/S9_5

S9.5 CELL 2: PASS


In [3]:

# ==========================================
# S9.5 - CELL 3: MODEL REGISTRY
# ==========================================

import json

# 1. Source utilities

SOURCE_MAP = registry.set_index("source_id")


def get_source(source_id):

    info = SOURCE_MAP.loc[source_id]

    path = SOURCE_ROOT / info["relative_path"]

    assert path.is_file()
    assert sha256_file(path) == info["sha256"]

    return path


def read_csv_source(source_id):

    return pd.read_csv(get_source(source_id))


def read_json_source(source_id):

    return json.loads(
        get_source(source_id).read_text(encoding="utf-8")
    )


# 2. Load source tables

b_grid = read_csv_source("SRC002")
b_val = read_csv_source("SRC004")

b_lock = read_json_source("SRC012")
b4_lock = read_json_source("SRC013")

b4_val = read_csv_source("SRC008")

assert len(b_grid) == 26
assert len(b_val) == 14
assert len(b4_val) == 6

print("[PASS] Source tables loaded")

# 3. Normalize model families

B_FAMILY_MAP = {
    "dummy": "Dummy",
    "logistic": "Logistic Regression",
    "decision_tree": "Decision Tree",
    "random_forest": "Random Forest",
    "primary": "XGBoost",
    "stacking": "Stacking"
}

C_FAMILY_MAP = {
    "B0_dummy": "Dummy",
    "B1_logistic": "Logistic Regression",
    "B2_random_forest": "Random Forest",
    "APPX_decision_tree": "Decision Tree",
    "X1_gaussian_nb": "Gaussian Naive Bayes",
    "X2_mlp": "MLP",
    "X3_svm_linear": "Linear SVM",
    "X4_logistic_SMOTE": "Logistic Regression",
    "X5_random_forest_SMOTE": "Random Forest"
}

# 4. Registry helper

model_rows = []


def add_model(
    model_id,
    owner,
    experiment_family,
    model_family,
    original_config,
    strategy,
    n_features,
    params_recorded,
    source_id,
    source_row,
    has_validation,
    params_source_id=""
):

    source = SOURCE_MAP.loc[source_id]

    model_rows.append({
        "model_id": model_id,
        "owner": owner,
        "experiment_family": experiment_family,
        "model_family": model_family,
        "original_config": original_config,
        "strategy": strategy,
        "n_features": int(n_features),
        "params_recorded": params_recorded,
        "params_source_id": params_source_id,
        "source_id": source_id,
        "source_row": int(source_row),
        "source_path": source["relative_path"],
        "source_sha256": source["sha256"],
        "has_validation": bool(has_validation),
        "status": "HISTORICAL_NON_TEST"
    })


# 5. B: Imbalance configurations

print("\n=== B: IMBALANCE CONFIGURATIONS ===")

assert not b_grid.duplicated(["model", "strategy"]).any()

config_map = {}

for config, model, strategy in b_lock["configs"]:

    key = (model, strategy)

    assert key not in config_map

    config_map[key] = config

val_keys = set(zip(b_val["model"], b_val["strategy"]))

assert set(config_map) == val_keys

for i, row in b_grid.iterrows():

    model = row["model"]
    strategy = row["strategy"]
    key = (model, strategy)

    params = b_lock["hyperparams"].get(model)

    if params is None:

        params_text = "Not recorded in lock"
        params_source = ""

    else:

        params_text = json.dumps(params, sort_keys=True)
        params_source = "SRC012"

    add_model(
        model_id=f"B__{model}__{strategy}",
        owner="B",
        experiment_family="imbalance_grid",
        model_family=B_FAMILY_MAP[model],
        original_config=config_map.get(key, model),
        strategy=strategy,
        n_features=21,
        params_recorded=params_text,
        source_id="SRC002",
        source_row=i + 1,
        has_validation=key in val_keys,
        params_source_id=params_source
    )

print("B imbalance configurations:", 26)

# 6. B: Feature selection

print("\n=== B: FEATURE CONFIGURATIONS ===")

assert set(b4_val["feature_set"]) == {
    "all_features",
    "selected",
    "k15_xgb_gain"
}

b4_unique = b4_val.drop_duplicates(
    subset=["feature_set"]
)

assert len(b4_unique) == 3

for i, row in b4_unique.iterrows():

    feature_set = row["feature_set"]

    feature_list = b4_lock["feature_sets"][feature_set]

    assert len(feature_list) == row["n_features"]
    assert row["strategy"] == b4_lock["strategy"]

    add_model(
        model_id=f"B__B4__{feature_set}",
        owner="B",
        experiment_family="feature_selection",
        model_family="XGBoost",
        original_config=feature_set,
        strategy=row["strategy"],
        n_features=row["n_features"],
        params_recorded=json.dumps(
            b4_lock["xgb_params"],
            sort_keys=True
        ),
        source_id="SRC008",
        source_row=i + 1,
        has_validation=True,
        params_source_id="SRC013"
    )

print("B feature configurations:", 3)

# 7. C: Baseline and Xie

print("\n=== C: BASELINE AND XIE ===")

c_sources = {
    "SRC014": "baseline",
    "SRC015": "baseline_appendix",
    "SRC020": "xie_appendix",
    "SRC021": "xie_appendix"
}

for source_id, family in c_sources.items():

    df = read_csv_source(source_id)

    assert set(df["split"]) == {"val"}
    assert set(df["seed"]) == {42}
    assert set(df["n_folds"]) == {3}
    assert not df["model"].duplicated().any()

    for i, row in df.iterrows():

        original_model = row["model"]

        add_model(
            model_id=f"C__{original_model}",
            owner="C",
            experiment_family=family,
            model_family=C_FAMILY_MAP[original_model],
            original_config=original_model,
            strategy=row["strategy"],
            n_features=21,
            params_recorded=str(row["params"]),
            source_id=source_id,
            source_row=i + 1,
            has_validation=True,
            params_source_id=source_id
        )

    print(source_id, "|", len(df), "models")

# 8. Consolidate registry

model_registry = pd.DataFrame(model_rows)

assert len(model_registry) == 38
assert model_registry["model_id"].is_unique
assert set(model_registry["owner"]) == {"B", "C"}
assert (model_registry["n_features"] > 0).all()

assert model_registry["source_id"].isin(
    registry["source_id"]
).all()

print("\n[PASS] Unique model identifiers")

# 9. Verify counts

assert (
    model_registry["owner"].value_counts().to_dict()
    == {"B": 29, "C": 9}
)

assert model_registry["has_validation"].sum() == 19

print("[PASS] Model registry counts")

# 10. F1/F2 links

c_f2 = read_csv_source("SRC014")
c_f1 = read_csv_source("SRC016")

assert set(c_f2["model"]) == set(c_f1["model"])

assert set(c_f2["threshold_rule"]) == {
    "thr_val_maxF2"
}

assert set(c_f1["threshold_rule"]) == {
    "thr_val_maxF1"
}

paired_threshold = c_f2.merge(
    c_f1,
    on=["model", "strategy", "params"],
    suffixes=("_F2", "_F1"),
    validate="one_to_one"
)

assert len(paired_threshold) == 3

for metric in ["AP", "auc_roc"]:

    np.testing.assert_allclose(
        paired_threshold[f"{metric}_F2"],
        paired_threshold[f"{metric}_F1"],
        rtol=0,
        atol=1e-12
    )

print("[PASS] F1/F2 configuration links")

# 11. Verify source-row provenance

for row in model_registry.itertuples(index=False):

    original = read_csv_source(row.source_id)

    assert 1 <= row.source_row <= len(original)

    record = original.iloc[row.source_row - 1]

    assert record["strategy"] == row.strategy

    if row.source_id == "SRC002":

        expected_id = (
            f"B__{record['model']}__{record['strategy']}"
        )

    elif row.source_id == "SRC008":

        expected_id = f"B__B4__{record['feature_set']}"

    else:

        expected_id = f"C__{record['model']}"

    assert expected_id == row.model_id

print("[PASS] Source-row provenance: 38/38")

# 12. Save

model_registry.to_csv(
    OUT / "model_registry.csv",
    index=False
)

assert (OUT / "model_registry.csv").is_file()

# 13. Display results

print("\n=== REGISTRY BY OWNER ===")

display(
    model_registry.groupby("owner")
    .size()
    .rename("n_configurations")
    .reset_index()
)

print("\n=== REGISTRY BY EXPERIMENT ===")

display(
    model_registry.groupby(["owner", "experiment_family"])
    .size()
    .rename("n_configurations")
    .reset_index()
)

print("\n=== MODEL REGISTRY ===")

display(
    model_registry[[
        "model_id",
        "owner",
        "model_family",
        "strategy",
        "n_features",
        "has_validation",
        "source_id",
        "source_row"
    ]]
)

print("\n=== REGISTRY SUMMARY ===")

print("Total configurations:", len(model_registry))

print(
    "Configurations with validation:",
    int(model_registry["has_validation"].sum())
)

print("Unique model IDs:", model_registry["model_id"].nunique())

print("Source provenance verified: 38/38")
print("Model training: NO")
print("Test evaluation: NO")
print("Output:", OUT / "model_registry.csv")

print("\nS9.5 CELL 3: PASS")


[PASS] Source tables loaded

=== B: IMBALANCE CONFIGURATIONS ===
B imbalance configurations: 26

=== B: FEATURE CONFIGURATIONS ===
B feature configurations: 3

=== C: BASELINE AND XIE ===
SRC014 | 3 models
SRC015 | 1 models
SRC020 | 3 models
SRC021 | 2 models

[PASS] Unique model identifiers
[PASS] Model registry counts
[PASS] F1/F2 configuration links
[PASS] Source-row provenance: 38/38

=== REGISTRY BY OWNER ===


,owner,n_configurations
0,B,29
1,C,9



=== REGISTRY BY EXPERIMENT ===


,owner,experiment_family,n_configurations
0,B,feature_selection,3
1,B,imbalance_grid,26
2,C,baseline,3
3,C,baseline_appendix,1
4,C,xie_appendix,5



=== MODEL REGISTRY ===


,model_id,owner,model_family,strategy,n_features,has_validation,source_id,source_row
0,B__dummy__none,B,Dummy,none,21,True,SRC002,1
1,B__logistic__none,B,Logistic Regression,none,21,True,SRC002,2
2,B__logistic__class_weight,B,Logistic Regression,class_weight,21,False,SRC002,3
3,B__logistic__smote,B,Logistic Regression,smote,21,False,SRC002,4
4,B__logistic__adasyn,B,Logistic Regression,adasyn,21,False,SRC002,5
5,B__logistic__undersample,B,Logistic Regression,undersample,21,False,SRC002,6
6,B__decision_tree__none,B,Decision Tree,none,21,True,SRC002,7
7,B__decision_tree__class_weight,B,Decision Tree,class_weight,21,False,SRC002,8
8,B__decision_tree__smote,B,Decision Tree,smote,21,False,SRC002,9
9,B__decision_tree__adasyn,B,Decision Tree,adasyn,21,False,SRC002,10



=== REGISTRY SUMMARY ===
Total configurations: 38
Configurations with validation: 19
Unique model IDs: 38
Source provenance verified: 38/38
Model training: NO
Test evaluation: NO
Output: /kaggle/working/S9_5/model_registry.csv

S9.5 CELL 3: PASS


In [4]:

# ==========================================
# S9.5 - CELL 4: CROSS-VALIDATION AGGREGATION
# ==========================================

import ast
import re

# 1. C tuning source map

C_TUNING = {
    "C__B1_logistic": "SRC017",
    "C__B2_random_forest": "SRC018",
    "C__APPX_decision_tree": "SRC019",
    "C__X1_gaussian_nb": "SRC022",
    "C__X2_mlp": "SRC023",
    "C__X3_svm_linear": "SRC024",
    "C__X4_logistic_SMOTE": "SRC025",
    "C__X5_random_forest_SMOTE": "SRC026"
}

M = model_registry.set_index("model_id")

cv_rows = []
tuning_rows = []

# 2. Provenance helper


def provenance(sid, i):

    src = SOURCE_MAP.loc[sid]

    return {
        "source_id": sid,
        "source_row": int(i) + 1,
        "source_path": src["relative_path"],
        "source_sha256": src["sha256"]
    }


# 3. Standardize CV records


def cv_record(mid, sid, i, row, kind):

    rec = {
        "model_id": mid,
        "owner": M.loc[mid, "owner"],
        "model_family": M.loc[mid, "model_family"],
        "strategy": M.loc[mid, "strategy"],
        "cv_kind": kind,
        "split": "train_cv",
        "cv_AP": float(row["cv_AP"]),
        "cv_AP_std": float(row["cv_AP_std"]),
        "cv_ROC_AUC": row.get("cv_auc", np.nan),
        "cv_recall_1": row.get("cv_recall_1", np.nan),
        "cv_f1_macro": row.get("cv_f1_macro", np.nan),
        **provenance(sid, i)
    }

    assert 0 <= rec["cv_AP"] <= 1
    assert np.isfinite(rec["cv_AP_std"])
    assert rec["cv_AP_std"] >= 0

    return rec


# 4. B imbalance CV

print("=== B: IMBALANCE GRID CV ===")

for i, row in read_csv_source("SRC002").iterrows():

    mid = f"B__{row['model']}__{row['strategy']}"

    cv_rows.append(
        cv_record(mid, "SRC002", i, row, "imbalance_grid")
    )

assert len(cv_rows) == 26

print("[PASS] B imbalance CV: 26/26")

# 5. C recorded tuning selections

print("\n=== C: HYPERPARAMETER TUNING ===")

for mid, sid in C_TUNING.items():

    t = read_csv_source(sid)

    assert len(t) == 10

    raw = str(M.loc[mid, "params_recorded"])

    cleaned = re.sub(
        r"np\.float64\(([-+0-9.eE]+)\)",
        r"\1",
        raw
    )

    params = ast.literal_eval(cleaned)

    keys = [
        k for k in t.columns
        if k not in {"cv_AP", "cv_AP_std", "seconds"}
    ]

    assert set(keys) == set(params)

    mask = np.ones(len(t), dtype=bool)

    for k in keys:

        v = params[k]

        if isinstance(v, tuple):

            same = (
                t[k]
                .map(ast.literal_eval)
                .map(lambda x: x == v)
                .to_numpy()
            )

        elif v is None:

            same = t[k].isna().to_numpy()

        else:

            same = np.isclose(
                t[k].to_numpy(dtype=float),
                float(v),
                rtol=1e-9,
                atol=0
            )

        mask &= same

    assert mask.sum() == 1, mid

    selected_i = t.index[mask][0]

    row = t.loc[selected_i]

    max_ap = t["cv_AP"].max()

    assert np.isclose(
        row["cv_AP"],
        max_ap,
        rtol=0,
        atol=1e-12
    ), mid

    rec = cv_record(
        mid,
        sid,
        selected_i,
        row,
        "hyperparameter_tuning"
    )

    rec.update({
        "params_recorded": raw,
        "params_source_id": M.loc[mid, "source_id"],
        "params_source_row": int(M.loc[mid, "source_row"]),
        "n_tied_max_AP_rounded": int(
            (t["cv_AP"] == max_ap).sum()
        )
    })

    cv_rows.append(rec)

    for i, candidate in t.iterrows():

        tuning_rows.append({
            "model_id": mid,
            **candidate.to_dict(),
            "is_recorded_selection": bool(i == selected_i),
            **provenance(sid, i)
        })

    print(
        f"[PASS] {mid}: "
        f"AP={row['cv_AP']:.4f}, "
        f"row={selected_i + 1}, "
        f"ties={rec['n_tied_max_AP_rounded']}"
    )

# 6. Consolidate CV

cv_summary = pd.DataFrame(cv_rows)

c_tuning_candidates = pd.DataFrame(tuning_rows)

assert len(cv_summary) == 34
assert cv_summary["model_id"].is_unique
assert len(c_tuning_candidates) == 80

assert c_tuning_candidates[
    "is_recorded_selection"
].sum() == 8

print("\n[PASS] Consolidated CV tables")

# 7. B tuning parameters

print("\n=== B: TUNING PARAMETERS ===")

b_tuning_recorded = read_csv_source(
    "SRC001"
).rename(columns={"Unnamed: 0": "model"})

assert set(b_tuning_recorded["model"]) == {
    "primary",
    "random_forest"
}

for i, row in b_tuning_recorded.iterrows():

    params = b_lock["hyperparams"][row["model"]]

    for k, v in params.items():

        assert np.isclose(row[k], v)

    for k, v in provenance("SRC001", i).items():

        b_tuning_recorded.loc[i, k] = v

    b_tuning_recorded.loc[i, "model_id"] = (
        f"B__{row['model']}__none"
    )

print("[PASS] B recorded tuning parameters: 2/2")

# 8. Missing CV coverage

missing_cv = model_registry.loc[
    ~model_registry["model_id"].isin(cv_summary["model_id"]),
    ["model_id", "owner", "experiment_family"]
].copy()

assert set(missing_cv["model_id"]) == {
    "C__B0_dummy",
    "B__B4__all_features",
    "B__B4__selected",
    "B__B4__k15_xgb_gain"
}

missing_cv["reason"] = np.where(
    missing_cv["owner"] == "B",
    "Feature CV handled separately",
    "No C dummy CV source in package"
)

# 9. Save outputs

tables = {
    "cv_summary": cv_summary,
    "c_tuning_candidates": c_tuning_candidates,
    "b_tuning_recorded": b_tuning_recorded,
    "cv_coverage_pending": missing_cv
}

for name, df in tables.items():

    path = OUT / f"{name}.csv"

    df.to_csv(path, index=False)

    assert path.is_file()

    print("[SAVED]", path.name, "| Rows:", len(df))

# 10. Display results

print("\n=== CV SUMMARY ===")

display(
    cv_summary[[
        "model_id",
        "cv_kind",
        "cv_AP",
        "cv_AP_std",
        "source_id",
        "source_row"
    ]]
)

print("\n=== COVERAGE ===")

display(missing_cv)

print("\n=== FINAL SUMMARY ===")

print("CV summary:", len(cv_summary))
print("C tuning candidates:", len(c_tuning_candidates))
print("B recorded tuning rows:", len(b_tuning_recorded))

print(
    "Recorded C selections:",
    int(c_tuning_candidates["is_recorded_selection"].sum())
)

print("Model training: NO")
print("Test evaluation: NO")

print("\nS9.5 CELL 4: PASS")


=== B: IMBALANCE GRID CV ===
[PASS] B imbalance CV: 26/26

=== C: HYPERPARAMETER TUNING ===
[PASS] C__B1_logistic: AP=0.4071, row=1, ties=1
[PASS] C__B2_random_forest: AP=0.4304, row=7, ties=1
[PASS] C__APPX_decision_tree: AP=0.4058, row=6, ties=1
[PASS] C__X1_gaussian_nb: AP=0.3494, row=7, ties=1
[PASS] C__X2_mlp: AP=0.4312, row=6, ties=1
[PASS] C__X3_svm_linear: AP=0.4062, row=10, ties=8
[PASS] C__X4_logistic_SMOTE: AP=0.4048, row=1, ties=1
[PASS] C__X5_random_forest_SMOTE: AP=0.4058, row=8, ties=1

[PASS] Consolidated CV tables

=== B: TUNING PARAMETERS ===
[PASS] B recorded tuning parameters: 2/2
[SAVED] cv_summary.csv | Rows: 34
[SAVED] c_tuning_candidates.csv | Rows: 80
[SAVED] b_tuning_recorded.csv | Rows: 2
[SAVED] cv_coverage_pending.csv | Rows: 4

=== CV SUMMARY ===


,model_id,cv_kind,cv_AP,cv_AP_std,source_id,source_row
0,B__dummy__none,imbalance_grid,0.1393,0.0000,SRC002,1
1,B__logistic__none,imbalance_grid,0.4065,0.0041,SRC002,2
2,B__logistic__class_weight,imbalance_grid,0.4050,0.0043,SRC002,3
3,B__logistic__smote,imbalance_grid,0.4042,0.0046,SRC002,4
4,B__logistic__adasyn,imbalance_grid,0.4028,0.0049,SRC002,5
5,B__logistic__undersample,imbalance_grid,0.4051,0.0039,SRC002,6
6,B__decision_tree__none,imbalance_grid,0.1895,0.0020,SRC002,7
7,B__decision_tree__class_weight,imbalance_grid,0.1849,0.0015,SRC002,8
8,B__decision_tree__smote,imbalance_grid,0.1874,0.0011,SRC002,9
9,B__decision_tree__adasyn,imbalance_grid,0.1875,0.0003,SRC002,10



=== COVERAGE ===


,model_id,owner,experiment_family,reason
26,B__B4__all_features,B,feature_selection,Feature CV handled separately
27,B__B4__selected,B,feature_selection,Feature CV handled separately
28,B__B4__k15_xgb_gain,B,feature_selection,Feature CV handled separately
29,C__B0_dummy,C,baseline,No C dummy CV source in package



=== FINAL SUMMARY ===
CV summary: 34
C tuning candidates: 80
B recorded tuning rows: 2
Recorded C selections: 8
Model training: NO
Test evaluation: NO

S9.5 CELL 4: PASS


In [5]:

# ==========================================
# S9.5 - CELL 5: VALIDATION AGGREGATION
# ==========================================

VAL_SOURCES = [
    "SRC004",
    "SRC008",
    "SRC014",
    "SRC015",
    "SRC016",
    "SRC020",
    "SRC021"
]

METRIC_COLUMNS = [
    "AP",
    "auc_roc",
    "accuracy",
    "balanced_acc",
    "f1_macro",
    "f1_class1",
    "f2_class1",
    "precision_1",
    "recall_1",
    "specificity",
    "mcc"
]

CM_COLUMNS = ["n_tp", "n_fp", "n_fn", "n_tn"]

ROUNDING_TOL = 5.01e-5

EXPECTED_VAL_SIZE = 38052
EXPECTED_POSITIVE = 5302
EXPECTED_NEGATIVE = 32750

validation_rows = []
recomputed_rows = []

# 1. Safe division


def safe_div(a, b):

    return float(a / b) if b != 0 else 0.0


# 2. Recompute metrics


def recompute_metrics(tp, fp, fn, tn):

    precision = safe_div(tp, tp + fp)

    recall = safe_div(tp, tp + fn)

    specificity = safe_div(tn, tn + fp)

    f1_positive = safe_div(
        2 * tp,
        2 * tp + fp + fn
    )

    f1_negative = safe_div(
        2 * tn,
        2 * tn + fp + fn
    )

    f2_positive = safe_div(
        5 * tp,
        5 * tp + 4 * fn + fp
    )

    mcc_denominator = np.sqrt(
        float(tp + fp)
        * (tp + fn)
        * (tn + fp)
        * (tn + fn)
    )

    return {
        "accuracy": safe_div(tp + tn, tp + fp + fn + tn),
        "balanced_acc": (recall + specificity) / 2,
        "f1_macro": (f1_positive + f1_negative) / 2,
        "f1_class1": f1_positive,
        "f2_class1": f2_positive,
        "precision_1": precision,
        "recall_1": recall,
        "specificity": specificity,
        "mcc": safe_div(
            tp * tn - fp * fn,
            mcc_denominator
        )
    }


# 3. Model identification


def get_validation_model_id(sid, row):

    if sid == "SRC004":

        return f"B__{row['model']}__{row['strategy']}"

    elif sid == "SRC008":

        return f"B__B4__{row['feature_set']}"

    else:

        return f"C__{row['model']}"


# 4. Read and verify validation

print("=== VALIDATION SOURCE AUDIT ===")

for sid in VAL_SOURCES:

    df = read_csv_source(sid)

    required = set(
        METRIC_COLUMNS + CM_COLUMNS
        + ["threshold", "threshold_rule"]
    )

    assert required.issubset(df.columns)

    if "split" in df.columns:

        assert set(df["split"]) == {"val"}

    if "seed" in df.columns:

        assert set(df["seed"]) == {42}

    if "n_folds" in df.columns:

        assert set(df["n_folds"]) == {3}

    for i, row in df.iterrows():

        mid = get_validation_model_id(sid, row)

        assert mid in M.index

        model_info = M.loc[mid]

        assert bool(model_info["has_validation"])
        assert row["strategy"] == model_info["strategy"]

        if sid == "SRC004":

            assert row["config"] == model_info["original_config"]

        if sid == "SRC008":

            assert row["n_features"] == model_info["n_features"]

        if model_info["owner"] == "C":

            assert str(row["params"]) == str(
                model_info["params_recorded"]
            )

        # Verify metric domains

        for metric in METRIC_COLUMNS:

            value = float(row[metric])

            assert np.isfinite(value)

            if metric == "mcc":

                assert -1 <= value <= 1

            else:

                assert 0 <= value <= 1

        # Verify confusion matrix

        counts = row[CM_COLUMNS].to_numpy(dtype=float)

        assert np.isfinite(counts).all()
        assert (counts >= 0).all()
        assert np.equal(counts, np.floor(counts)).all()

        tp, fp, fn, tn = map(int, counts)

        assert tp + fp + fn + tn == EXPECTED_VAL_SIZE

        assert tp + fn == EXPECTED_POSITIVE
        assert fp + tn == EXPECTED_NEGATIVE

        threshold = float(row["threshold"])

        assert 0 <= threshold <= 1

        rule = row["threshold_rule"]

        assert rule in {
            "thr_0.5",
            "thr_val_maxF1",
            "thr_val_maxF2"
        }

        if rule == "thr_0.5":

            assert np.isclose(threshold, 0.5)

        # Recompute nine metrics

        recalculated = recompute_metrics(tp, fp, fn, tn)

        for metric, value in recalculated.items():

            recorded = float(row[metric])

            error = abs(recorded - value)

            assert error <= ROUNDING_TOL, (
                sid, i + 1, metric, error
            )

            recomputed_rows.append({
                "model_id": mid,
                "threshold_rule": rule,
                "metric": metric,
                "recorded_value": recorded,
                "recomputed_value": value,
                "absolute_error": error,
                "status": "PASS",
                **provenance(sid, i)
            })

        record = {
            "model_id": mid,
            "owner": model_info["owner"],
            "experiment_family": model_info["experiment_family"],
            "model_family": model_info["model_family"],
            "original_config": model_info["original_config"],
            "strategy": model_info["strategy"],
            "n_features": int(model_info["n_features"]),
            "params_recorded": model_info["params_recorded"],
            "split": "validation",
            "split_evidence": (
                "explicit_csv_column"
                if "split" in df.columns
                else "source_filename_and_context"
            ),
            "threshold_rule": rule,
            "threshold": threshold,
            "n_validation": EXPECTED_VAL_SIZE,
            "n_positive": EXPECTED_POSITIVE,
            "n_negative": EXPECTED_NEGATIVE,
            "status": "HISTORICAL_NON_TEST",
            **{
                metric: float(row[metric])
                for metric in METRIC_COLUMNS
            },
            "n_tp": tp,
            "n_fp": fp,
            "n_fn": fn,
            "n_tn": tn,
            **provenance(sid, i)
        }

        validation_rows.append(record)

    print(f"[PASS] {sid}: {len(df)} rows")

# 5. Consolidate

validation_summary = pd.DataFrame(validation_rows)

validation_metrics_recomputed = pd.DataFrame(
    recomputed_rows
)

assert len(validation_summary) == 32
assert len(validation_metrics_recomputed) == 288

assert not validation_summary.duplicated(
    ["model_id", "threshold_rule"]
).any()

expected_ids = set(
    model_registry.loc[
        model_registry["has_validation"],
        "model_id"
    ]
)

assert set(validation_summary["model_id"]) == expected_ids

print("\n[PASS] Validation coverage: 19/19")
print("[PASS] Confusion matrix metrics: 288/288")

# 6. Separate threshold rules

validation_main = validation_summary.loc[
    validation_summary["threshold_rule"] == "thr_val_maxF2"
].copy()

validation_default = validation_summary.loc[
    validation_summary["threshold_rule"] == "thr_0.5"
].copy()

validation_f1 = validation_summary.loc[
    validation_summary["threshold_rule"] == "thr_val_maxF1"
].copy()

assert len(validation_main) == 19
assert len(validation_default) == 10
assert len(validation_f1) == 3

assert validation_main["model_id"].is_unique
assert set(validation_main["model_id"]) == expected_ids

print("[PASS] Threshold rule separation")

# 7. Threshold-independent metrics

for mid, group in validation_summary.groupby("model_id"):

    for metric in ["AP", "auc_roc"]:

        np.testing.assert_allclose(
            group[metric],
            group[metric].iloc[0],
            rtol=0,
            atol=1e-12
        )

print(
    "[PASS] AP and ROC-AUC invariant "
    "across recorded thresholds"
)

# 8. Compare F2/F1 rules

for reference, metric in [
    (validation_default, "f2_class1"),
    (validation_f1, "f1_class1")
]:

    paired = validation_main.merge(
        reference,
        on="model_id",
        suffixes=("_F2", "_reference"),
        validate="one_to_one"
    )

    if metric == "f2_class1":

        assert (
            paired[f"{metric}_F2"] + ROUNDING_TOL
            >= paired[f"{metric}_reference"]
        ).all()

    else:

        assert (
            paired[f"{metric}_reference"] + ROUNDING_TOL
            >= paired[f"{metric}_F2"]
        ).all()

print("[PASS] Recorded threshold comparisons")

# 9. Document overlapping B records

overlap_audit = []

for rule in ["thr_0.5", "thr_val_maxF2"]:

    a = validation_summary.loc[
        (
            validation_summary["model_id"]
            == "B__primary__class_weight"
        )
        & (validation_summary["threshold_rule"] == rule)
    ].iloc[0]

    b = validation_summary.loc[
        (
            validation_summary["model_id"]
            == "B__B4__all_features"
        )
        & (validation_summary["threshold_rule"] == rule)
    ].iloc[0]

    compare_columns = (
        METRIC_COLUMNS + CM_COLUMNS + ["threshold"]
    )

    np.testing.assert_allclose(
        a[compare_columns].to_numpy(dtype=float),
        b[compare_columns].to_numpy(dtype=float),
        rtol=0,
        atol=1e-12
    )

    overlap_audit.append({
        "model_id_A": a["model_id"],
        "model_id_B": b["model_id"],
        "threshold_rule": rule,
        "source_id_A": a["source_id"],
        "source_row_A": a["source_row"],
        "source_id_B": b["source_id"],
        "source_row_B": b["source_row"],
        "recorded_metrics_match": True,
        "prediction_identity_verified": False,
        "interpretation": (
            "Matching reported metrics; "
            "not independent evidence"
        )
    })

validation_overlap_audit = pd.DataFrame(overlap_audit)

print("[PASS] B all-feature reference overlap documented")

# 10. Save tables

tables = {
    "validation_summary": validation_summary,
    "validation_main_F2": validation_main,
    "validation_default_05": validation_default,
    "validation_sensitivity_F1": validation_f1,
    "validation_metrics_recomputed":
        validation_metrics_recomputed,
    "validation_overlap_audit": validation_overlap_audit
}

for name, df in tables.items():

    path = OUT / f"{name}.csv"

    df.to_csv(path, index=False)

    print("[SAVED]", path.name, "| Rows:", len(df))

# 11. Display results

print("\n=== MAIN VALIDATION RESULTS: MAX F2 ===")

display(
    validation_main[[
        "model_id",
        "AP",
        "auc_roc",
        "threshold",
        "f2_class1",
        "precision_1",
        "recall_1",
        "specificity",
        "source_id",
        "source_row"
    ]]
)

print("\n=== THRESHOLD RULE COVERAGE ===")

display(
    validation_summary.groupby("threshold_rule")
    .size()
    .rename("n_rows")
    .reset_index()
)

print("\n=== METRIC RECOMPUTATION ===")

display(
    validation_metrics_recomputed.groupby("metric")[
        "absolute_error"
    ]
    .agg(["count", "max"])
    .reset_index()
)

max_error = float(
    validation_metrics_recomputed["absolute_error"].max()
)

print("\n=== FINAL SUMMARY ===")

print("Validation records:", len(validation_summary))
print("Max F2 records:", len(validation_main))
print("Default 0.5 records:", len(validation_default))
print("Max F1 records:", len(validation_f1))
print("Unique configurations:", len(expected_ids))

print(
    "Recomputed metrics:",
    len(validation_metrics_recomputed),
    "/ 288"
)

print("Maximum absolute error:", f"{max_error:.10f}")

print("Documented overlap rows:", len(validation_overlap_audit))

print("Model training: NO")
print("Test evaluation: NO")

print("\nS9.5 CELL 5: PASS")


=== VALIDATION SOURCE AUDIT ===
[PASS] SRC004: 14 rows
[PASS] SRC008: 6 rows
[PASS] SRC014: 3 rows
[PASS] SRC015: 1 rows
[PASS] SRC016: 3 rows
[PASS] SRC020: 3 rows
[PASS] SRC021: 2 rows

[PASS] Validation coverage: 19/19
[PASS] Confusion matrix metrics: 288/288
[PASS] Threshold rule separation
[PASS] AP and ROC-AUC invariant across recorded thresholds
[PASS] Recorded threshold comparisons
[PASS] B all-feature reference overlap documented
[SAVED] validation_summary.csv | Rows: 32
[SAVED] validation_main_F2.csv | Rows: 19
[SAVED] validation_default_05.csv | Rows: 10
[SAVED] validation_sensitivity_F1.csv | Rows: 3
[SAVED] validation_metrics_recomputed.csv | Rows: 288
[SAVED] validation_overlap_audit.csv | Rows: 2

=== MAIN VALIDATION RESULTS: MAX F2 ===


,model_id,AP,auc_roc,threshold,f2_class1,precision_1,recall_1,specificity,source_id,source_row
1,B__dummy__none,0.1393,0.5000,0.0100,0.4474,0.1393,1.0000,0.0000,SRC004,2
3,B__logistic__none,0.4039,0.8209,0.1090,0.5987,0.2806,0.8355,0.6533,SRC004,4
5,B__decision_tree__none,0.1923,0.6003,0.4010,0.3261,0.2893,0.3369,0.8660,SRC004,6
7,B__random_forest__none,0.4231,0.8238,0.1387,0.5978,0.2946,0.8050,0.6880,SRC004,8
9,B__primary__none,0.4258,0.8285,0.1139,0.6033,0.2837,0.8399,0.6567,SRC004,10
11,B__primary__class_weight,0.4256,0.8281,0.4604,0.6039,0.2897,0.8286,0.6711,SRC004,12
13,B__stacking__class_weight,0.4274,0.8282,0.1090,0.6038,0.2878,0.8323,0.6666,SRC004,14
15,B__B4__all_features,0.4256,0.8281,0.4604,0.6039,0.2897,0.8286,0.6711,SRC008,2
17,B__B4__selected,0.4226,0.8259,0.4505,0.6010,0.2828,0.8361,0.6567,SRC008,4
19,B__B4__k15_xgb_gain,0.4258,0.8280,0.4604,0.6034,0.2898,0.8272,0.6718,SRC008,6



=== THRESHOLD RULE COVERAGE ===


,threshold_rule,n_rows
0,thr_0.5,10
1,thr_val_maxF1,3
2,thr_val_maxF2,19



=== METRIC RECOMPUTATION ===


,metric,count,max
0,accuracy,32,0.000050
1,balanced_acc,32,0.000050
2,f1_class1,32,0.000050
3,f1_macro,32,0.000046
4,f2_class1,32,0.000049
5,mcc,32,0.000049
6,precision_1,32,0.000049
7,recall_1,32,0.000046
8,specificity,32,0.000048



=== FINAL SUMMARY ===
Validation records: 32
Max F2 records: 19
Default 0.5 records: 10
Max F1 records: 3
Unique configurations: 19
Recomputed metrics: 288 / 288
Maximum absolute error: 0.0000499394
Documented overlap rows: 2
Model training: NO
Test evaluation: NO

S9.5 CELL 5: PASS


In [6]:

# ==========================================
# S9.5 - CELL 6: SENSITIVITY ANALYSIS
# ==========================================

import time

cell6_start = time.perf_counter()

sensitivity_checks = {}


def sensitivity_pass(name):

    sensitivity_checks[name] = "PASS"

    elapsed = time.perf_counter() - cell6_start

    print(
        f"[PASS] {name} | Elapsed: {elapsed:.2f}s",
        flush=True
    )


def source_table(sid):

    df = read_csv_source(sid).copy()

    for i in df.index:

        for key, value in provenance(sid, i).items():

            df.loc[i, key] = value

    return df


# ==========================================
# 1. IMBALANCE STRATEGY
# ==========================================

print("=== IMBALANCE ANALYSIS ===", flush=True)

imbalance_summary = cv_summary.loc[
    cv_summary["cv_kind"] == "imbalance_grid"
].copy()

assert len(imbalance_summary) == 26

baseline_ap = (
    imbalance_summary.loc[
        imbalance_summary["strategy"] == "none"
    ]
    .set_index("model_family")["cv_AP"]
)

imbalance_summary["reference_AP_none"] = (
    imbalance_summary["model_family"].map(baseline_ap)
)

imbalance_summary["delta_AP_vs_none"] = (
    imbalance_summary["cv_AP"]
    - imbalance_summary["reference_AP_none"]
)

assert imbalance_summary["reference_AP_none"].notna().all()

sensitivity_pass("S01_imbalance_coverage")

# Verify recorded strategy

strategy_selection = source_table("SRC003")

chosen = strategy_selection.iloc[0]

assert chosen["chosen_strategy"] == b_lock["best_strategy"]

primary_grid = read_csv_source("SRC002")

primary_grid = primary_grid.loc[
    primary_grid["model"] == "primary"
].copy()

max_macro_f1 = primary_grid["cv_f1_macro"].max()

eligible = primary_grid.loc[
    primary_grid["cv_f1_macro"]
    >= max_macro_f1 - 0.01 - 1e-12
]

selected_strategy = eligible.sort_values(
    ["cv_AP", "cv_recall_1"],
    ascending=[False, False]
).iloc[0]

assert selected_strategy["strategy"] == chosen["chosen_strategy"]

assert np.isclose(
    selected_strategy["cv_AP"],
    chosen["cv_AP"]
)

assert np.isclose(
    selected_strategy["cv_recall_1"],
    chosen["cv_recall_1"]
)

sensitivity_pass("S02_recorded_strategy_rule")


# ==========================================
# 2. FEATURE SELECTION CV
# ==========================================

print("\n=== FEATURE SELECTION ===", flush=True)

feature_selection_summary = source_table("SRC006")

assert len(feature_selection_summary) == 25

assert not feature_selection_summary.duplicated(
    ["method", "k"]
).any()

assert set(feature_selection_summary["k"]) == {
    5, 10, 15, 21
}

assert (
    feature_selection_summary["k"]
    == feature_selection_summary["n_features_kept"]
).all()

full = feature_selection_summary.loc[
    (feature_selection_summary["method"] == "baseline")
    & (feature_selection_summary["k"] == 21)
].iloc[0]

full_AP = float(full["cv_AP"])

feature_selection_summary["delta_AP_vs_full21"] = (
    feature_selection_summary["cv_AP"] - full_AP
)

feature_selection_summary["within_AP_tolerance"] = (
    feature_selection_summary["delta_AP_vs_full21"]
    >= -0.01 - 1e-12
)

sensitivity_pass("S03_feature_cv_coverage")

# Verify recorded feature selection

feature_selected_config = source_table("SRC007")

selected = feature_selected_config.iloc[0]

assert selected["selected_method"] == (
    b4_lock["selected_method"]
)

assert selected["k"] == b4_lock["k"]
assert selected["strategy"] == b4_lock["strategy"]

matched = feature_selection_summary.loc[
    (
        feature_selection_summary["method"]
        == selected["selected_method"]
    )
    & (feature_selection_summary["k"] == selected["k"])
]

assert len(matched) == 1

matched = matched.iloc[0]

for metric in ["cv_AP", "cv_recall_1", "cv_f1_macro"]:

    assert np.isclose(
        matched[metric],
        selected[metric]
    )

assert np.isclose(selected["cv_AP_full21"], full_AP)

assert np.isclose(
    selected["delta_AP_vs_full"],
    matched["delta_AP_vs_full21"]
)

assert matched["within_AP_tolerance"]

eligible_k = feature_selection_summary.loc[
    feature_selection_summary["within_AP_tolerance"],
    "k"
]

assert eligible_k.min() == selected["k"]

sensitivity_pass("S04_recorded_feature_selection")


# ==========================================
# 3. FEATURE SELECTION VALIDATION
# ==========================================

feature_validation_summary = validation_main.loc[
    validation_main["experiment_family"]
    == "feature_selection"
].copy()

assert len(feature_validation_summary) == 3

assert set(feature_validation_summary["n_features"]) == {
    10, 15, 21
}

full_val_AP = float(
    feature_validation_summary.loc[
        feature_validation_summary["n_features"] == 21,
        "AP"
    ].iloc[0]
)

feature_validation_summary["delta_AP_vs_full21"] = (
    feature_validation_summary["AP"] - full_val_AP
)

sensitivity_pass("S05_feature_validation")


# ==========================================
# 4. SELECTED FEATURES AND IMPORTANCE
# ==========================================

f10 = source_table("SRC009")
f15 = source_table("SRC010")

assert len(f10) == 10
assert len(f15) == 15

assert f10["feature"].is_unique
assert f15["feature"].is_unique

assert f10["feature"].tolist() == (
    b4_lock["feature_sets"]["selected"]
)

assert f15["feature"].tolist() == (
    b4_lock["feature_sets"]["k15_xgb_gain"]
)

assert set(f10["feature"]).issubset(f15["feature"])

selected_feature_registry = pd.concat(
    [f10, f15],
    ignore_index=True
)

sensitivity_pass("S06_selected_feature_lists")

permutation_importance = source_table("SRC011")

assert len(permutation_importance) == 10
assert permutation_importance["feature"].is_unique

assert set(permutation_importance["feature"]) == (
    set(f10["feature"])
)

assert (permutation_importance["imp_std"] >= 0).all()

assert np.isfinite(
    permutation_importance[
        ["imp_mean", "imp_std"]
    ].to_numpy()
).all()

sensitivity_pass("S07_permutation_importance")


# ==========================================
# 5. FEATURE METHOD OVERLAP
# ==========================================

feature_overlap_rows = []

for method_a, method_b in [
    ("xgb_gain", "rfe_xgb"),
    ("variance", "corr_dedup")
]:

    left = feature_selection_summary.loc[
        feature_selection_summary["method"] == method_a
    ]

    right = feature_selection_summary.loc[
        feature_selection_summary["method"] == method_b
    ]

    paired = left.merge(
        right,
        on="k",
        suffixes=("_A", "_B"),
        validate="one_to_one"
    )

    assert len(paired) == 4

    for _, row in paired.iterrows():

        assert np.isclose(
            row["cv_AP_A"],
            row["cv_AP_B"],
            rtol=0,
            atol=1e-12
        )

        feature_overlap_rows.append({
            "method_A": method_a,
            "method_B": method_b,
            "k": row["k"],
            "cv_AP_A": row["cv_AP_A"],
            "cv_AP_B": row["cv_AP_B"],
            "source_id": "SRC006",
            "source_row_A": row["source_row_A"],
            "source_row_B": row["source_row_B"],
            "recorded_AP_match": True,
            "prediction_identity_verified": False
        })

feature_method_overlap = pd.DataFrame(
    feature_overlap_rows
)

assert len(feature_method_overlap) == 8

sensitivity_pass("S08_feature_method_overlap")


# ==========================================
# 6. THRESHOLD SENSITIVITY
# ==========================================

print("\n=== THRESHOLD SENSITIVITY ===", flush=True)

threshold_rows = []

for ref_table, reference_name in [
    (validation_default, "thr_0.5"),
    (validation_f1, "thr_val_maxF1")
]:

    paired = validation_main.merge(
        ref_table,
        on="model_id",
        suffixes=("_F2", "_ref"),
        validate="one_to_one"
    )

    for _, row in paired.iterrows():

        record = {
            "model_id": row["model_id"],
            "owner": row["owner_F2"],
            "comparison": f"{reference_name}_to_maxF2",
            "threshold_reference": row["threshold_ref"],
            "threshold_F2": row["threshold_F2"],
            "source_id_reference": row["source_id_ref"],
            "source_row_reference": row["source_row_ref"],
            "source_id_F2": row["source_id_F2"],
            "source_row_F2": row["source_row_F2"]
        }

        for metric in [
            "AP",
            "auc_roc",
            "f1_class1",
            "f2_class1",
            "precision_1",
            "recall_1",
            "specificity",
            "mcc",
            "n_tp",
            "n_fp",
            "n_fn",
            "n_tn"
        ]:

            record[f"{metric}_reference"] = row[f"{metric}_ref"]

            record[f"{metric}_F2"] = row[f"{metric}_F2"]

            record[f"delta_{metric}"] = (
                row[f"{metric}_F2"] - row[f"{metric}_ref"]
            )

        threshold_rows.append(record)

threshold_sensitivity_summary = pd.DataFrame(
    threshold_rows
)

assert len(threshold_sensitivity_summary) == 13

assert np.allclose(
    threshold_sensitivity_summary["delta_AP"],
    0,
    rtol=0,
    atol=1e-12
)

assert np.allclose(
    threshold_sensitivity_summary["delta_auc_roc"],
    0,
    rtol=0,
    atol=1e-12
)

sensitivity_pass("S09_threshold_sensitivity")


# ==========================================
# 7. GROUP-VECTOR CV
# ==========================================

print("\n=== GROUP-VECTOR CV ===", flush=True)

b_group = source_table("SRC005")
c_group = source_table("SRC027")
c_folds = source_table("SRC028")
c_comparison = source_table("SRC029")

s7_config = read_json_source("SRC034")

assert s7_config["scope"] == "train_only"
assert s7_config["seed"] == 42
assert s7_config["n_folds"] == 3

assert not s7_config["retuning"]
assert not s7_config["holdout_test_used_for_cv"]

assert len(b_group) == 2
assert len(c_group) == 4
assert len(c_folds) == 12
assert len(c_comparison) == 2

group_rows = []

# B: recorded summary

method_map = {
    "stratified (mac dinh)": "StratifiedKFold",
    "group_split (theo nhom vector trung)": "GroupKFold"
}

for _, row in b_group.iterrows():

    group_rows.append({
        "model_id": "B__primary__class_weight",
        "owner": "B",
        "method": method_map[row["split_type"]],
        "cv_AP": row["cv_AP"],
        "cv_AP_std": row["cv_AP_std"],
        "cv_AUC": np.nan,
        "cv_AUC_std": np.nan,
        "n_groups": row["n_groups"],
        "fold_level_available": False,
        "source_id": row["source_id"],
        "source_row": row["source_row"],
        "source_sha256": row["source_sha256"]
    })

# C: recompute from fold-level results

for _, row in c_group.iterrows():

    subset = c_folds.loc[
        (c_folds["model"] == row["model"])
        & (c_folds["method"] == row["method"])
    ]

    assert len(subset) == 3
    assert set(subset["fold"]) == {1, 2, 3}

    assert (
        subset["n_train"] + subset["n_eval"]
        == s7_config["n_train"]
    ).all()

    actual = [
        subset["AP"].mean(),
        subset["AP"].std(ddof=0),
        subset["ROC_AUC"].mean(),
        subset["ROC_AUC"].std(ddof=0)
    ]

    expected = [
        row["cv_AP"],
        row["cv_AP_std"],
        row["cv_AUC"],
        row["cv_AUC_std"]
    ]

    np.testing.assert_allclose(
        actual,
        expected,
        rtol=0,
        atol=1e-9
    )

    group_rows.append({
        "model_id": f"C__{row['model']}",
        "owner": "C",
        "method": row["method"],
        "cv_AP": row["cv_AP"],
        "cv_AP_std": row["cv_AP_std"],
        "cv_AUC": row["cv_AUC"],
        "cv_AUC_std": row["cv_AUC_std"],
        "n_groups": s7_config["n_groups"],
        "fold_level_available": True,
        "source_id": row["source_id"],
        "source_row": row["source_row"],
        "source_sha256": row["source_sha256"]
    })

group_cv_summary = pd.DataFrame(group_rows)

assert len(group_cv_summary) == 6

sensitivity_pass("S10_group_cv_summary")


# ==========================================
# 8. GROUP CV DELTAS
# ==========================================

group_delta_rows = []

for mid, subset in group_cv_summary.groupby("model_id"):

    assert len(subset) == 2

    strat = subset.loc[
        subset["method"] == "StratifiedKFold"
    ].iloc[0]

    group = subset.loc[
        subset["method"] == "GroupKFold"
    ].iloc[0]

    delta = group["cv_AP"] - strat["cv_AP"]

    if strat["owner"] == "C":

        original = c_comparison.loc[
            c_comparison["model"] == mid.removeprefix("C__")
        ].iloc[0]

        assert np.isclose(
            delta,
            original["delta_AP_group_minus_stratified"],
            rtol=0,
            atol=1e-9
        )

        historical = cv_summary.loc[
            cv_summary["model_id"] == mid,
            "cv_AP"
        ].iloc[0]

        assert np.isclose(
            round(strat["cv_AP"], 4),
            historical
        )

    group_delta_rows.append({
        "model_id": mid,
        "owner": strat["owner"],
        "stratified_cv_AP": strat["cv_AP"],
        "group_cv_AP": group["cv_AP"],
        "delta_AP": delta,
        "delta_AUC": group["cv_AUC"] - strat["cv_AUC"],
        "source_id_stratified": strat["source_id"],
        "source_row_stratified": strat["source_row"],
        "source_id_group": group["source_id"],
        "source_row_group": group["source_row"]
    })

group_cv_comparison = pd.DataFrame(group_delta_rows)

assert len(group_cv_comparison) == 3

sensitivity_pass("S11_group_cv_deltas")


# ==========================================
# 9. GROUP OVERLAP AUDIT
# ==========================================

group_overlap_summary = source_table("SRC030")
group_fold_audit = source_table("SRC031")

assert len(group_overlap_summary) == 2
assert len(group_fold_audit) == 6

for _, row in group_overlap_summary.iterrows():

    subset = group_fold_audit.loc[
        group_fold_audit["method"] == row["method"]
    ]

    assert len(subset) == 3

    assert subset["n_eval"].sum() == s7_config["n_train"]

    assert subset["n_eval"].sum() == row["n_eval_total"]

    assert subset["shared_eval_rows"].sum() == (
        row["shared_eval_rows_total"]
    )

    assert subset["index_overlap"].sum() == (
        row["index_overlap_total"]
    )

    assert np.isclose(
        subset["shared_groups"].mean(),
        row["shared_groups_mean"]
    )

    assert np.isclose(
        subset["shared_eval_rows"].sum()
        / subset["n_eval"].sum(),
        row["shared_eval_rate"],
        rtol=0,
        atol=1e-9
    )

    assert row["index_overlap_total"] == 0

    if row["method"] == "GroupKFold":

        assert row["shared_eval_rows_total"] == 0

        assert (subset["shared_groups"] == 0).all()

sensitivity_pass("S12_group_overlap_audit")


# ==========================================
# 10. SAVE OUTPUTS
# ==========================================

tables = {
    "imbalance_summary":
        imbalance_summary,

    "imbalance_selected_strategy":
        strategy_selection,

    "feature_selection_summary":
        feature_selection_summary,

    "feature_selected_config":
        feature_selected_config,

    "feature_validation_summary":
        feature_validation_summary,

    "selected_feature_registry":
        selected_feature_registry,

    "permutation_importance":
        permutation_importance,

    "feature_method_overlap":
        feature_method_overlap,

    "threshold_sensitivity_summary":
        threshold_sensitivity_summary,

    "group_cv_summary":
        group_cv_summary,

    "group_cv_comparison":
        group_cv_comparison,

    "group_overlap_summary":
        group_overlap_summary,

    "group_fold_audit":
        group_fold_audit
}

print("\n=== SAVED TABLES ===", flush=True)

for name, df in tables.items():

    df.to_csv(
        OUT / f"{name}.csv",
        index=False
    )

    print(
        "[SAVED]",
        f"{name}.csv",
        "| Rows:",
        len(df),
        flush=True
    )

assert len(sensitivity_checks) == 12

with open(
    OUT / "sensitivity_checks.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        sensitivity_checks,
        f,
        indent=2
    )


# ==========================================
# 11. DISPLAY KEY RESULTS
# ==========================================

print("\n=== FEATURE SELECTION SUMMARY ===")

display(
    feature_selection_summary[[
        "method",
        "k",
        "cv_AP",
        "delta_AP_vs_full21",
        "within_AP_tolerance"
    ]]
)

print("\n=== THRESHOLD SENSITIVITY ===")

display(
    threshold_sensitivity_summary[[
        "model_id",
        "comparison",
        "delta_AP",
        "delta_f2_class1",
        "delta_precision_1",
        "delta_recall_1"
    ]]
)

print("\n=== GROUP CV COMPARISON ===")

display(group_cv_comparison)

print("\n=== GROUP OVERLAP ===")

display(group_overlap_summary)

print("\n=== FINAL SUMMARY ===")

print("Imbalance configurations:", len(imbalance_summary))

print("Feature CV records:", len(feature_selection_summary))

print(
    "Feature validation records:",
    len(feature_validation_summary)
)

print(
    "Threshold comparisons:",
    len(threshold_sensitivity_summary)
)

print("Group CV records:", len(group_cv_summary))

print("Group CV comparisons:", len(group_cv_comparison))

print("Tables saved:", len(tables))

print("Sensitivity QA:", len(sensitivity_checks), "/ 12")

print("Model training: NO")
print("Test evaluation: NO")

print(
    "Cell 6 execution time:",
    f"{time.perf_counter() - cell6_start:.2f} seconds"
)

print("\nS9.5 CELL 6: PASS")


=== IMBALANCE ANALYSIS ===
[PASS] S01_imbalance_coverage | Elapsed: 0.01s
[PASS] S02_recorded_strategy_rule | Elapsed: 0.02s

=== FEATURE SELECTION ===
[PASS] S03_feature_cv_coverage | Elapsed: 0.04s
[PASS] S04_recorded_feature_selection | Elapsed: 0.05s
[PASS] S05_feature_validation | Elapsed: 0.06s
[PASS] S06_selected_feature_lists | Elapsed: 0.08s
[PASS] S07_permutation_importance | Elapsed: 0.09s
[PASS] S08_feature_method_overlap | Elapsed: 0.10s

=== THRESHOLD SENSITIVITY ===
[PASS] S09_threshold_sensitivity | Elapsed: 0.11s

=== GROUP-VECTOR CV ===
[PASS] S10_group_cv_summary | Elapsed: 0.16s
[PASS] S11_group_cv_deltas | Elapsed: 0.16s
[PASS] S12_group_overlap_audit | Elapsed: 0.18s

=== SAVED TABLES ===
[SAVED] imbalance_summary.csv | Rows: 26
[SAVED] imbalance_selected_strategy.csv | Rows: 1
[SAVED] feature_selection_summary.csv | Rows: 25
[SAVED] feature_selected_config.csv | Rows: 1
[SAVED] feature_validation_summary.csv | Rows: 3
[SAVED] selected_feature_registry.csv | Rows:

,method,k,cv_AP,delta_AP_vs_full21,within_AP_tolerance
0,baseline,21,0.4360,0.0000,True
1,variance,21,0.4360,0.0000,True
2,variance,15,0.4280,-0.0080,True
3,variance,10,0.4136,-0.0224,False
4,variance,5,0.3530,-0.0830,False
5,corr_dedup,21,0.4360,0.0000,True
6,corr_dedup,15,0.4280,-0.0080,True
7,corr_dedup,10,0.4136,-0.0224,False
8,corr_dedup,5,0.3530,-0.0830,False
9,mutual_info,21,0.4360,0.0000,True



=== THRESHOLD SENSITIVITY ===


,model_id,comparison,delta_AP,delta_f2_class1,delta_precision_1,delta_recall_1
0,B__dummy__none,thr_0.5_to_maxF2,0.0,0.4474,0.1393,1.0000
1,B__logistic__none,thr_0.5_to_maxF2,0.0,0.4106,-0.2563,0.6737
2,B__decision_tree__none,thr_0.5_to_maxF2,0.0,0.0000,0.0000,0.0000
3,B__random_forest__none,thr_0.5_to_maxF2,0.0,0.4723,-0.2975,0.7001
4,B__primary__none,thr_0.5_to_maxF2,0.0,0.4126,-0.2777,0.6762
5,B__primary__class_weight,thr_0.5_to_maxF2,0.0,0.0041,-0.0144,0.0361
6,B__stacking__class_weight,thr_0.5_to_maxF2,0.0,0.4328,-0.2803,0.6867
7,B__B4__all_features,thr_0.5_to_maxF2,0.0,0.0041,-0.0144,0.0361
8,B__B4__selected,thr_0.5_to_maxF2,0.0,0.0033,-0.0172,0.0411
9,B__B4__k15_xgb_gain,thr_0.5_to_maxF2,0.0,0.0041,-0.0143,0.0358



=== GROUP CV COMPARISON ===


,model_id,owner,stratified_cv_AP,group_cv_AP,delta_AP,delta_AUC,source_id_stratified,source_row_stratified,source_id_group,source_row_group
0,B__primary__class_weight,B,0.436000,0.435000,-0.001000,NaN,SRC005,1.0,SRC005,2.0
1,C__B1_logistic,C,0.407068,0.407090,0.000022,-0.000069,SRC027,2.0,SRC027,1.0
2,C__B2_random_forest,C,0.430360,0.430687,0.000327,-0.000072,SRC027,4.0,SRC027,3.0



=== GROUP OVERLAP ===


,method,n_eval_total,shared_eval_rows_total,shared_groups_mean,index_overlap_total,shared_eval_rate,source_id,source_row,source_path,source_sha256
0,GroupKFold,177576,0,0.000000,0,0.000000,SRC030,1.0,C/report/results/C_group_cv/s7_overlap_summary...,44375cfcceae0b8c59b0ee303f5462d8f6cb03d5143592...
1,StratifiedKFold,177576,19078,4292.666667,0,0.107436,SRC030,2.0,C/report/results/C_group_cv/s7_overlap_summary...,44375cfcceae0b8c59b0ee303f5462d8f6cb03d5143592...



=== FINAL SUMMARY ===
Imbalance configurations: 26
Feature CV records: 25
Feature validation records: 3
Threshold comparisons: 13
Group CV records: 6
Group CV comparisons: 3
Tables saved: 13
Sensitivity QA: 12 / 12
Model training: NO
Test evaluation: NO
Cell 6 execution time: 0.23 seconds

S9.5 CELL 6: PASS


In [11]:

# ==========================================
# S9.5 - CELL 7: BUILD AGGREGATION SCRIPT
# ==========================================

import ast
import sys
import re
import textwrap
import py_compile
import subprocess

from IPython import get_ipython

SCRIPT_PATH = OUT / "c_aggregate_results.py"

build_checks = {}


def build_pass(name):

    build_checks[name] = "PASS"

    print(f"[PASS] {name}", flush=True)


# ==========================================
# 1. VERIFY PREVIOUS RESULTS
# ==========================================

print("=== PREVIOUS CELL VERIFICATION ===")

EXPECTED_TABLES = {

    # Cell 2
    "source_integrity_audit.csv": 34,
    "source_registry.csv": 34,

    # Cell 3
    "model_registry.csv": 38,

    # Cell 4
    "cv_summary.csv": 34,
    "c_tuning_candidates.csv": 80,
    "b_tuning_recorded.csv": 2,
    "cv_coverage_pending.csv": 4,

    # Cell 5
    "validation_summary.csv": 32,
    "validation_main_F2.csv": 19,
    "validation_default_05.csv": 10,
    "validation_sensitivity_F1.csv": 3,
    "validation_metrics_recomputed.csv": 288,
    "validation_overlap_audit.csv": 2,

    # Cell 6
    "imbalance_summary.csv": 26,
    "imbalance_selected_strategy.csv": 1,
    "feature_selection_summary.csv": 25,
    "feature_selected_config.csv": 1,
    "feature_validation_summary.csv": 3,
    "selected_feature_registry.csv": 25,
    "permutation_importance.csv": 10,
    "feature_method_overlap.csv": 8,
    "threshold_sensitivity_summary.csv": 13,
    "group_cv_summary.csv": 6,
    "group_cv_comparison.csv": 3,
    "group_overlap_summary.csv": 2,
    "group_fold_audit.csv": 6
}

assert len(EXPECTED_TABLES) == 26

for filename, expected_rows in EXPECTED_TABLES.items():

    path = OUT / filename

    assert path.is_file(), (
        f"Missing output: {filename}"
    )

    df = pd.read_csv(path)

    assert len(df) == expected_rows, (
        filename,
        len(df),
        expected_rows
    )

build_pass("B01_previous_output_coverage_26_of_26")


# Verify Cell 6 QA

sensitivity_disk = json.loads(
    (OUT / "sensitivity_checks.json")
    .read_text(encoding="utf-8")
)

assert len(sensitivity_disk) == 12

assert all(
    value == "PASS"
    for value in sensitivity_disk.values()
)

build_pass("B02_sensitivity_QA_12_of_12")


# ==========================================
# 2. RECOVER EXECUTED NOTEBOOK CELLS
# ==========================================

print("\n=== RECOVER NOTEBOOK SOURCE ===")

ip = get_ipython()

assert ip is not None

history = ip.history_manager.input_hist_raw

cell_sources = {}

pattern = re.compile(
    r"^\s*#\s*=+\s*\n"
    r"# S9\.5 - CELL ([1-6]):"
)

for execution_id, source in enumerate(history):

    match = pattern.match(source)

    if match:

        cell_number = int(match.group(1))

        # Keep the most recent execution
        cell_sources[cell_number] = {

            "execution_id": execution_id,

            "source": source
        }


assert set(cell_sources) == set(
    range(1, 7)
), (
    "Cannot recover all six cells from "
    "the current session. Run Cells 1-6 again."
)

execution_ids = [

    cell_sources[i]["execution_id"]

    for i in range(1, 7)
]

assert execution_ids == sorted(execution_ids), (
    "Cells were not executed sequentially. "
    "Run Cells 1-6 in order."
)

for i in range(1, 7):

    source = cell_sources[i]["source"]

    ast.parse(source)

    print(
        f"Cell {i} | "
        f"Execution ID: "
        f"{cell_sources[i]['execution_id']} | "
        f"Lines: {len(source.splitlines())}"
    )

build_pass("B03_notebook_source_recovery")


# ==========================================
# 3. ADAPT CELL 1 FOR CLI
# ==========================================

print("\n=== ADAPT SOURCE FOR CLI ===")

original_cell1 = cell_sources[1]["source"]

replacements = {

    'INPUT_ROOT = Path("/kaggle/input")':

        "INPUT_ROOT = Path(input_root).resolve()",

    'OUT = Path("/kaggle/working/S9_5")':

        "OUT = Path(out_dir).resolve()",

    "from IPython.display import display":

        (
            "def display(obj):\n"
            "    if hasattr(obj, 'to_string'):\n"
            "        print(obj.to_string("
            "index=False, max_rows=12))\n"
            "    else:\n"
            "        print(obj)"
        )
}

adapted_cell1 = original_cell1

for old, new in replacements.items():

    assert adapted_cell1.count(old) == 1, (
        f"Cannot adapt source: {old}"
    )

    adapted_cell1 = adapted_cell1.replace(
        old,
        new,
        1
    )

ast.parse(adapted_cell1)

build_pass("B04_cli_path_adaptation")


# ==========================================
# 4. BUILD STANDALONE SCRIPT
# ==========================================

print("\n=== BUILD PYTHON SCRIPT ===")

script_header = '''"""
S9.5 - Historical CV and validation aggregation.

Generated from the six notebook cells used
for source auditing and result aggregation.

Scope:
- Existing CV and validation results only.
- No model training.
- No raw BRFSS data.
- No test or FINAL performance evaluation.

The historical source package is verified
using its pinned SHA-256 hashes.
"""

import argparse
from pathlib import Path


def aggregate(input_root, out_dir):

    if not __debug__:
        raise RuntimeError(
            "Run without Python optimization (-O). "
            "The pipeline requires assertions."
        )

    input_root = Path(input_root).resolve()
    out_dir = Path(out_dir).resolve()

    if not input_root.is_dir():
        raise FileNotFoundError(input_root)

    if out_dir.is_relative_to(input_root):
        raise ValueError(
            "Output must be outside input directory."
        )

    if out_dir.exists():

        if not out_dir.is_dir():
            raise ValueError(
                "Output path is not a directory."
            )

        if any(out_dir.iterdir()):
            raise FileExistsError(
                "Output directory must be empty."
            )

'''


# Fix Path variable scope inside aggregate()

header_start = "def aggregate(input_root, out_dir):\n"

fixed_start = (
    header_start
    + "\n    from pathlib import Path\n"
)

if fixed_start not in script_header:

    assert header_start in script_header

    script_header = script_header.replace(
        header_start,
        fixed_start,
        1
    )

assert fixed_start in script_header

print("[PASS] Path scope fix applied")

script_parts = [script_header]


for i in range(1, 7):

    if i == 1:

        source = adapted_cell1

    else:

        source = cell_sources[i]["source"]

    script_parts.append(
        f"\n    # BEGIN NOTEBOOK CELL {i}\n"
    )

    script_parts.append(
        textwrap.indent(
            source.rstrip(),
            "    "
        )
    )

    script_parts.append(
        f"\n    # END NOTEBOOK CELL {i}\n"
    )


# CLI entry point

script_footer = '''

    return OUT


def main():

    parser = argparse.ArgumentParser(
        description=(
            "Aggregate historical CV and "
            "validation results."
        )
    )

    parser.add_argument(
        "--input-root",
        required=True,
        help="Directory containing verified input package"
    )

    parser.add_argument(
        "--out-dir",
        required=True,
        help="New or empty output directory"
    )

    args = parser.parse_args()

    result_dir = aggregate(
        input_root=args.input_root,
        out_dir=args.out_dir
    )

    print("\\nAggregation output:", result_dir)
    print("S9.5 AGGREGATION: PASS")


if __name__ == "__main__":
    main()
'''

script_parts.append(script_footer)

script_source = "\n".join(script_parts)

# Validate complete source before saving

ast.parse(script_source)

compile(
    script_source,
    str(SCRIPT_PATH),
    "exec"
)

SCRIPT_PATH.write_text(
    script_source,
    encoding="utf-8"
)

assert SCRIPT_PATH.is_file()

assert SCRIPT_PATH.stat().st_size > 0

build_pass("B05_standalone_script_created")


# ==========================================
# 5. VERIFY SOURCE PRESERVATION
# ==========================================

saved_source = SCRIPT_PATH.read_text(
    encoding="utf-8"
)

for i in range(2, 7):

    original = cell_sources[i]["source"]

    expected = textwrap.indent(
        original.rstrip(),
        "    "
    )

    assert expected in saved_source, (
        f"Cell {i} source differs"
    )

assert textwrap.indent(
    adapted_cell1.rstrip(),
    "    "
) in saved_source

build_pass("B06_notebook_logic_preserved")


# ==========================================
# 6. PYTHON SYNTAX CHECK
# ==========================================

py_compile.compile(
    str(SCRIPT_PATH),
    doraise=True
)

build_pass("B07_python_syntax")


# ==========================================
# 7. CLI HELP SMOKE TEST
# ==========================================

print("\n=== CLI HELP TEST ===")

result = subprocess.run(

    [
        sys.executable,
        str(SCRIPT_PATH),
        "--help"
    ],

    capture_output=True,

    text=True,

    timeout=30
)

assert result.returncode == 0, result.stderr

assert "--input-root" in result.stdout
assert "--out-dir" in result.stdout

print(result.stdout)

build_pass("B08_cli_help")


# ==========================================
# 8. SAVE BUILD PROVENANCE
# ==========================================

assert len(build_checks) == 8

assert all(
    status == "PASS"
    for status in build_checks.values()
)

build_report = {

    "task": "S9_5_aggregation_script_build",

    "scope": "HISTORICAL_NON_TEST",

    "script_name": SCRIPT_PATH.name,

    "script_sha256": sha256_file(SCRIPT_PATH),

    "script_bytes": SCRIPT_PATH.stat().st_size,

    "script_lines": len(
        saved_source.splitlines()
    ),

    "source_cells": [

        {
            "cell": i,

            "execution_id":
                cell_sources[i]["execution_id"],

            "original_source_sha256":
                hashlib.sha256(
                    cell_sources[i]["source"]
                    .encode("utf-8")
                ).hexdigest()
        }

        for i in range(1, 7)
    ],

    "adapted_cell1_sha256":

        hashlib.sha256(
            adapted_cell1.encode("utf-8")
        ).hexdigest(),

    "adaptations": [

        "Parameterized input directory",

        "Parameterized output directory",

        "Replaced IPython display with console output",

        "Wrapped notebook cells in aggregate function",

        "Added command-line interface",

        "Require new or empty output directory",

        "Reject Python optimized execution"
    ],

    "expected_csv_outputs": EXPECTED_TABLES,

    "expected_json_outputs": [
        "sensitivity_checks.json"
    ],

    "checks": build_checks,

    "total_checks": 8,

    "total_passed": len(build_checks),

    "full_cli_execution_verified": False,

    "model_training": False,

    "test_evaluation": False
}

with open(
    OUT / "aggregation_build_report.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        build_report,
        f,
        indent=2
    )


# ==========================================
# 9. FINAL SUMMARY
# ==========================================

print("\n=== SCRIPT BUILD SUMMARY ===")

print("Notebook cells recovered: 6/6")

print("Existing CSV outputs verified: 26/26")

print(
    "Sensitivity QA:",
    len(sensitivity_disk),
    "/ 12"
)

print(
    "Build QA:",
    len(build_checks),
    "/ 8"
)

print("Script:", SCRIPT_PATH)

print(
    "Script bytes:",
    SCRIPT_PATH.stat().st_size
)

print(
    "Script lines:",
    len(saved_source.splitlines())
)

print(
    "Script SHA-256:",
    sha256_file(SCRIPT_PATH)
)

print("Full CLI execution: PENDING")

print("Model training: NO")
print("Test evaluation: NO")

print("\nS9.5 CELL 7: PASS")


=== PREVIOUS CELL VERIFICATION ===
[PASS] B01_previous_output_coverage_26_of_26
[PASS] B02_sensitivity_QA_12_of_12

=== RECOVER NOTEBOOK SOURCE ===
Cell 1 | Execution ID: 1 | Lines: 67
Cell 2 | Execution ID: 2 | Lines: 198
Cell 3 | Execution ID: 3 | Lines: 409
Cell 4 | Execution ID: 4 | Lines: 311
Cell 5 | Execution ID: 5 | Lines: 510
Cell 6 | Execution ID: 6 | Lines: 797
[PASS] B03_notebook_source_recovery

=== ADAPT SOURCE FOR CLI ===
[PASS] B04_cli_path_adaptation

=== BUILD PYTHON SCRIPT ===
[PASS] Path scope fix applied
[PASS] B05_standalone_script_created
[PASS] B06_notebook_logic_preserved
[PASS] B07_python_syntax

=== CLI HELP TEST ===
usage: c_aggregate_results.py [-h] --input-root INPUT_ROOT --out-dir OUT_DIR

Aggregate historical CV and validation results.

options:
  -h, --help            show this help message and exit
  --input-root INPUT_ROOT
                        Directory containing verified input package
  --out-dir OUT_DIR     New or empty output directory

[PASS] 

In [12]:

# ==========================================
# S9.5 - CELL 8: REPRODUCIBILITY AND QA V2
# ==========================================

import sys
import json
import shutil
import tempfile
import subprocess
import time

qa_start = time.perf_counter()

QA_DIR = OUT / "qa"
QA_DIR.mkdir(parents=True, exist_ok=True)

final_checks = {}
reproduction_rows = []
metric_audit_rows = []
negative_test_rows = []


def final_pass(name):

    final_checks[name] = "PASS"

    elapsed = time.perf_counter() - qa_start

    print(
        f"[PASS] {name} | Elapsed: {elapsed:.2f}s",
        flush=True
    )


# ==========================================
# 1. VERIFY SCRIPT IDENTITY
# ==========================================

print("=== SCRIPT IDENTITY ===", flush=True)

SCRIPT_PATH = OUT / "c_aggregate_results.py"

BUILD_REPORT_PATH = (
    OUT / "aggregation_build_report.json"
)

assert SCRIPT_PATH.is_file()
assert BUILD_REPORT_PATH.is_file()

build_disk = json.loads(
    BUILD_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)

# V2: Read the new hash generated by Cell 7.
# No hard-coded hash from the previous build.

EXPECTED_SCRIPT_HASH = build_disk[
    "script_sha256"
]

ACTUAL_SCRIPT_HASH = sha256_file(
    SCRIPT_PATH
)

assert ACTUAL_SCRIPT_HASH == (
    EXPECTED_SCRIPT_HASH
), "Script differs from Cell 7 build report"

assert build_disk["script_name"] == (
    SCRIPT_PATH.name
)

assert build_disk["script_bytes"] == (
    SCRIPT_PATH.stat().st_size
)

assert build_disk["total_checks"] == 8
assert build_disk["total_passed"] == 8

assert len(build_disk["checks"]) == 8

assert all(
    value == "PASS"
    for value in build_disk["checks"].values()
)

assert len(EXPECTED_TABLES) == 26

assert build_disk[
    "expected_csv_outputs"
] == EXPECTED_TABLES

# Verify that the Path scope fix was applied.

script_text = SCRIPT_PATH.read_text(
    encoding="utf-8"
)

assert (
    "def aggregate(input_root, out_dir):"
    in script_text
)

function_body = script_text.split(
    "def aggregate(input_root, out_dir):",
    1
)[1]

assert function_body.index(
    "from pathlib import Path"
) < function_body.index(
    "input_root = Path(input_root)"
), "Cell 7 Path scope fix is missing"

print("Expected SHA-256:", EXPECTED_SCRIPT_HASH)
print("Actual SHA-256:  ", ACTUAL_SCRIPT_HASH)
print("Path scope fix: PRESENT")

final_pass("Q01_script_identity")


# ==========================================
# 2. CLI EXECUTION HELPER
# ==========================================

def run_cli(
    input_root,
    out_dir,
    optimized=False
):

    command = [sys.executable]

    if optimized:

        command.append("-O")

    command.extend([
        str(SCRIPT_PATH),

        "--input-root",
        str(input_root),

        "--out-dir",
        str(out_dir)
    ])

    return subprocess.run(
        command,
        capture_output=True,
        text=True,
        timeout=120
    )


def save_cli_log(name, result):

    text = (
        f"Return code: {result.returncode}\n\n"
        f"=== STDOUT ===\n"
        f"{result.stdout}\n\n"
        f"=== STDERR ===\n"
        f"{result.stderr}"
    )

    (QA_DIR / f"{name}.txt").write_text(
        text,
        encoding="utf-8"
    )


# ==========================================
# 3. FULL CLI REPRODUCTION
# ==========================================

print("\n=== FULL CLI REPRODUCTION ===", flush=True)

with tempfile.TemporaryDirectory(
    prefix="s9_reproduction_"
) as temp:

    temp = Path(temp)

    cli_out = temp / "reproduced_results"

    result = run_cli(
        SOURCE_ROOT,
        cli_out
    )

    save_cli_log(
        "full_cli_execution",
        result
    )

    if result.returncode != 0:

        print("\n=== CLI STDOUT ===")
        print(result.stdout)

        print("\n=== CLI STDERR ===")
        print(result.stderr)

        raise RuntimeError(
            "Standalone aggregation script failed. "
            "See full_cli_execution.txt"
        )

    assert (
        "S9.5 AGGREGATION: PASS"
        in result.stdout
    )

    for i in range(1, 7):

        assert (
            f"S9.5 CELL {i}: PASS"
            in result.stdout
        )

    final_pass("Q02_full_cli_execution")


    # ======================================
    # 4. COMPARE ALL CSV OUTPUTS
    # ======================================

    print("\n=== CSV REPRODUCIBILITY ===")

    for filename, expected_rows in (
        EXPECTED_TABLES.items()
    ):

        original = OUT / filename

        reproduced = cli_out / filename

        assert original.is_file(), filename

        assert reproduced.is_file(), filename

        hash_original = sha256_file(
            original
        )

        hash_reproduced = sha256_file(
            reproduced
        )

        original_df = pd.read_csv(
            original
        )

        reproduced_df = pd.read_csv(
            reproduced
        )

        assert len(original_df) == (
            expected_rows
        )

        assert len(reproduced_df) == (
            expected_rows
        )

        pd.testing.assert_frame_equal(
            original_df,
            reproduced_df,
            check_exact=True
        )

        assert hash_original == hash_reproduced, (
            filename,
            hash_original,
            hash_reproduced
        )

        reproduction_rows.append({

            "filename": filename,

            "n_rows": expected_rows,

            "notebook_sha256": hash_original,

            "cli_sha256": hash_reproduced,

            "identical_bytes": True,

            "status": "PASS"
        })

        print(
            f"[MATCH] {filename}",
            flush=True
        )

    assert len(reproduction_rows) == 26

    final_pass(
        "Q03_csv_reproduction_26_of_26"
    )


    # ======================================
    # 5. VERIFY JSON AND OUTPUT COVERAGE
    # ======================================

    json_name = "sensitivity_checks.json"

    original_json = OUT / json_name

    reproduced_json = cli_out / json_name

    assert original_json.is_file()
    assert reproduced_json.is_file()

    j1 = json.loads(
        original_json.read_text(
            encoding="utf-8"
        )
    )

    j2 = json.loads(
        reproduced_json.read_text(
            encoding="utf-8"
        )
    )

    assert j1 == j2
    assert len(j1) == 12

    assert all(
        value == "PASS"
        for value in j1.values()
    )

    assert sha256_file(original_json) == (
        sha256_file(reproduced_json)
    )

    reproduction_rows.append({

        "filename": json_name,

        "n_rows": None,

        "notebook_sha256":
            sha256_file(original_json),

        "cli_sha256":
            sha256_file(reproduced_json),

        "identical_bytes": True,

        "status": "PASS"
    })

    expected_names = set(
        EXPECTED_TABLES
    ) | {json_name}

    actual_names = {
        p.name
        for p in cli_out.iterdir()
        if p.is_file()
    }

    assert actual_names == expected_names

    assert len(actual_names) == 27

    final_pass(
        "Q04_json_and_output_coverage"
    )


# ==========================================
# 6. INDEPENDENT SOURCE-ROW METRIC AUDIT
# ==========================================

print("\n=== SOURCE-ROW METRIC AUDIT ===")

cv_disk = pd.read_csv(
    OUT / "cv_summary.csv"
)

val_disk = pd.read_csv(
    OUT / "validation_summary.csv"
)

assert len(cv_disk) == 34
assert len(val_disk) == 32


def audit_source_metrics(
    table,
    table_name,
    metrics
):

    for _, row in table.iterrows():

        sid = row["source_id"]

        source_row = int(
            row["source_row"]
        )

        source_path = get_source(sid)

        assert row["source_sha256"] == (
            sha256_file(source_path)
        )

        original = pd.read_csv(
            source_path
        )

        assert 1 <= source_row <= len(
            original
        )

        record = original.iloc[
            source_row - 1
        ]

        for metric in metrics:

            stored = float(
                row[metric]
            )

            source_value = float(
                record[metric]
            )

            assert np.isfinite(stored)
            assert np.isfinite(source_value)

            error = abs(
                stored - source_value
            )

            assert error <= 1e-12, (
                table_name,
                row["model_id"],
                metric,
                error
            )

            metric_audit_rows.append({

                "table": table_name,

                "model_id": row["model_id"],

                "threshold_rule": row.get(
                    "threshold_rule", ""
                ),

                "metric": metric,

                "source_id": sid,

                "source_row": source_row,

                "source_sha256":
                    row["source_sha256"],

                "source_value":
                    source_value,

                "aggregated_value":
                    stored,

                "absolute_error": error,

                "status": "PASS"
            })


# CV: 34 rows x 2 metrics = 68

audit_source_metrics(
    cv_disk,
    "cv_summary",
    ["cv_AP", "cv_AP_std"]
)


# Validation: 32 rows x 16 fields = 512

audit_source_metrics(
    val_disk,
    "validation_summary",
    METRIC_COLUMNS + CM_COLUMNS + ["threshold"]
)

assert len(metric_audit_rows) == 580

final_pass(
    "Q05_source_metric_values_580_of_580"
)


# ==========================================
# 7. NEGATIVE TESTING HELPERS
# ==========================================

print("\n=== NEGATIVE TESTS ===", flush=True)


def expect_cli_failure(
    name,
    input_root,
    out_dir,
    expected_message=None,
    optimized=False
):

    result = run_cli(
        input_root,
        out_dir,
        optimized=optimized
    )

    save_cli_log(
        name,
        result
    )

    assert result.returncode != 0, (
        f"{name}: Invalid execution accepted"
    )

    assert (
        "S9.5 AGGREGATION: PASS"
        not in result.stdout
    )

    combined_output = (
        result.stdout + result.stderr
    )

    if expected_message is not None:

        assert (
            expected_message
            in combined_output
        ), combined_output

    negative_test_rows.append({

        "test": name,

        "returncode": result.returncode,

        "expected_message":
            expected_message,

        "invalid_execution_rejected": True,

        "status": "PASS"
    })

    final_pass(name)


# ==========================================
# 8. CORRUPTED SOURCE AND REGISTRY
# ==========================================

# Modify temporary copies only.
# Never modify original Kaggle inputs.

with tempfile.TemporaryDirectory(
    prefix="s9_negative_tests_"
) as temp:

    temp = Path(temp)

    copied_input = temp / "input_copy"

    shutil.copytree(
        SOURCE_ROOT,
        copied_input
    )


    # ======================================
    # Q06: CORRUPTED SOURCE FILE
    # ======================================

    source_info = SOURCE_MAP.loc["SRC002"]

    changed_file = (
        copied_input
        / source_info["relative_path"]
    )

    original_bytes = (
        changed_file.read_bytes()
    )

    with open(changed_file, "ab") as f:

        f.write(b"\nTAMPERED_QA_TEST\n")

    assert sha256_file(changed_file) != (
        source_info["sha256"]
    )

    expect_cli_failure(

        "Q06_corrupted_source_rejected",

        copied_input,

        temp / "bad_source_output",

        expected_message="Hash mismatch"
    )

    # Restore the temporary file.

    changed_file.write_bytes(
        original_bytes
    )

    assert sha256_file(changed_file) == (
        source_info["sha256"]
    )


    # ======================================
    # Q07: CORRUPTED SOURCE REGISTRY
    # ======================================

    copied_registry = (
        copied_input / "source_registry.csv"
    )

    registry_bytes = (
        copied_registry.read_bytes()
    )

    with open(copied_registry, "ab") as f:

        f.write(b"\nTAMPERED_REGISTRY\n")

    assert sha256_file(copied_registry) != (
        EXPECTED_REGISTRY_HASH
    )

    expect_cli_failure(

        "Q07_corrupted_registry_rejected",

        copied_input,

        temp / "bad_registry_output",

        expected_message="AssertionError"
    )

    # Restore the temporary registry.

    copied_registry.write_bytes(
        registry_bytes
    )

    assert sha256_file(copied_registry) == (
        EXPECTED_REGISTRY_HASH
    )


    # ======================================
    # Q08: NONEMPTY OUTPUT DIRECTORY
    # ======================================

    nonempty_out = (
        temp / "nonempty_output"
    )

    nonempty_out.mkdir()

    marker = (
        nonempty_out / "old_result.txt"
    )

    marker.write_text(
        "Preserve existing output",
        encoding="utf-8"
    )

    expect_cli_failure(

        "Q08_nonempty_output_rejected",

        copied_input,

        nonempty_out,

        expected_message=(
            "Output directory must be empty"
        )
    )

    assert marker.read_text(
        encoding="utf-8"
    ) == "Preserve existing output"

    assert len(
        list(nonempty_out.iterdir())
    ) == 1


    # ======================================
    # Q09: OUTPUT INSIDE INPUT DIRECTORY
    # ======================================

    nested_out = (
        copied_input / "nested_output"
    )

    expect_cli_failure(

        "Q09_nested_output_rejected",

        copied_input,

        nested_out,

        expected_message=(
            "Output must be outside input directory"
        )
    )

    assert not nested_out.exists()


    # ======================================
    # Q10: PYTHON OPTIMIZATION MODE
    # ======================================

    expect_cli_failure(

        "Q10_optimized_python_rejected",

        copied_input,

        temp / "optimized_output",

        expected_message=(
            "Run without Python optimization"
        ),

        optimized=True
    )


# ==========================================
# 9. VERIFY ORIGINAL INPUTS UNCHANGED
# ==========================================

print("\n=== ORIGINAL INPUT VERIFICATION ===")

assert sha256_file(REGISTRY_PATH) == (
    EXPECTED_REGISTRY_HASH
)

for row in registry.itertuples(
    index=False
):

    path = (
        SOURCE_ROOT / row.relative_path
    )

    assert sha256_file(path) == row.sha256

    assert path.stat().st_size == row.bytes

print(
    "[PASS] Original sources unchanged: 34/34"
)

# Ensure the tested script was not modified.

assert sha256_file(SCRIPT_PATH) == (
    EXPECTED_SCRIPT_HASH
)


# ==========================================
# 10. SAVE QA EVIDENCE
# ==========================================

reproduction_audit = pd.DataFrame(
    reproduction_rows
)

source_metric_audit = pd.DataFrame(
    metric_audit_rows
)

negative_test_audit = pd.DataFrame(
    negative_test_rows
)

assert len(reproduction_audit) == 27

assert len(source_metric_audit) == 580

assert len(negative_test_audit) == 5

assert len(final_checks) == 10

assert all(
    value == "PASS"
    for value in final_checks.values()
)

reproduction_audit.to_csv(
    QA_DIR / "reproduction_audit.csv",
    index=False
)

source_metric_audit.to_csv(
    QA_DIR / "source_metric_audit.csv",
    index=False
)

negative_test_audit.to_csv(
    QA_DIR / "negative_test_audit.csv",
    index=False
)


# ==========================================
# 11. CONSOLIDATED QA REPORT
# ==========================================

max_source_error = float(
    source_metric_audit[
        "absolute_error"
    ].max()
)

qa_report = {

    "task": "S9_5_result_aggregation",

    "qa_version": "V2",

    "scope": "HISTORICAL_NON_TEST",

    "status": "AGGREGATION_QA_PASS",

    "script_sha256":
        sha256_file(SCRIPT_PATH),

    "script_hash_reference":
        "aggregation_build_report.json",

    "build_report_sha256":
        sha256_file(BUILD_REPORT_PATH),

    "path_scope_fix_verified": True,

    "source_files_verified": 34,

    "model_configurations": 38,

    "cv_records": 34,

    "validation_records": 32,

    "confusion_matrix_metrics_verified": 288,

    "sensitivity_checks_passed": 12,

    "build_checks_passed": 8,

    "final_checks": final_checks,

    "final_checks_passed": 10,

    "final_checks_total": 10,

    "reproduced_csv_files": 26,

    "reproduced_json_files": 1,

    "byte_identical_files": 27,

    "source_metric_values_verified": 580,

    "maximum_source_metric_error":
        max_source_error,

    "negative_tests_passed": 5,

    "original_sources_unchanged": True,

    "full_cli_execution_verified": True,

    "tested_input_mode":
        "extracted_directory",

    "model_training_performed": False,

    "raw_brfss_used": False,

    "test_evaluation_performed": False,

    "final_evaluation_performed": False,

    "team_cross_review_completed": False,

    "limitations": [

        "Reproduction verifies aggregation, "
        "not model retraining.",

        "AP and ROC-AUC are checked against "
        "stored source values, not recalculated "
        "from original prediction scores.",

        "Historical B code and lock version "
        "mismatch remains unresolved.",

        "A4/A5 official results are not included.",

        "CLI ZIP-input execution was not "
        "independently tested in this cell.",

        "Script identity is checked against "
        "the current Cell 7 build report, "
        "not an external immutable hash."
    ]
}

with open(
    QA_DIR / "aggregation_qa_report.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        qa_report,
        f,
        indent=2
    )


# ==========================================
# 12. FINAL SUMMARY
# ==========================================

print("\n=== REPRODUCIBILITY AUDIT ===")

display(
    reproduction_audit[[
        "filename",
        "n_rows",
        "identical_bytes",
        "status"
    ]]
)

print("\n=== NEGATIVE TEST AUDIT ===")

display(negative_test_audit)

print("\n=== FINAL QA SUMMARY ===")

print("Path scope fix: VERIFIED")

print(
    "Script SHA-256:",
    sha256_file(SCRIPT_PATH)
)

print("Full CLI execution: PASS")

print("CSV reproduction: 26/26")

print("JSON reproduction: 1/1")

print("Byte-identical outputs: 27/27")

print("Source metric values: 580/580")

print(
    "Maximum source metric error:",
    f"{max_source_error:.12f}"
)

print("Negative tests: 5/5")

print("Original sources unchanged: 34/34")

print(
    "Final QA:",
    len(final_checks),
    "/ 10"
)

print("Model training: NO")
print("Test evaluation: NO")

print("QA output:", QA_DIR)

print(
    "Cell 8 execution time:",
    f"{time.perf_counter() - qa_start:.2f} seconds"
)

print("\nS9.5 CELL 8: PASS")


=== SCRIPT IDENTITY ===
Expected SHA-256: 8abc4bbdcec57113fd2e6ed667e6dcf123900b64b3a6ddfd7357f72af9dc5a2b
Actual SHA-256:   8abc4bbdcec57113fd2e6ed667e6dcf123900b64b3a6ddfd7357f72af9dc5a2b
Path scope fix: PRESENT
[PASS] Q01_script_identity | Elapsed: 0.00s

=== FULL CLI REPRODUCTION ===
[PASS] Q02_full_cli_execution | Elapsed: 1.28s

=== CSV REPRODUCIBILITY ===
[MATCH] source_integrity_audit.csv
[MATCH] source_registry.csv
[MATCH] model_registry.csv
[MATCH] cv_summary.csv
[MATCH] c_tuning_candidates.csv
[MATCH] b_tuning_recorded.csv
[MATCH] cv_coverage_pending.csv
[MATCH] validation_summary.csv
[MATCH] validation_main_F2.csv
[MATCH] validation_default_05.csv
[MATCH] validation_sensitivity_F1.csv
[MATCH] validation_metrics_recomputed.csv
[MATCH] validation_overlap_audit.csv
[MATCH] imbalance_summary.csv
[MATCH] imbalance_selected_strategy.csv
[MATCH] feature_selection_summary.csv
[MATCH] feature_selected_config.csv
[MATCH] feature_validation_summary.csv
[MATCH] selected_feature_registr

,filename,n_rows,identical_bytes,status
0,source_integrity_audit.csv,34.0,True,PASS
1,source_registry.csv,34.0,True,PASS
2,model_registry.csv,38.0,True,PASS
3,cv_summary.csv,34.0,True,PASS
4,c_tuning_candidates.csv,80.0,True,PASS
5,b_tuning_recorded.csv,2.0,True,PASS
6,cv_coverage_pending.csv,4.0,True,PASS
7,validation_summary.csv,32.0,True,PASS
8,validation_main_F2.csv,19.0,True,PASS
9,validation_default_05.csv,10.0,True,PASS



=== NEGATIVE TEST AUDIT ===


,test,returncode,expected_message,invalid_execution_rejected,status
0,Q06_corrupted_source_rejected,1,Hash mismatch,True,PASS
1,Q07_corrupted_registry_rejected,1,AssertionError,True,PASS
2,Q08_nonempty_output_rejected,1,Output directory must be empty,True,PASS
3,Q09_nested_output_rejected,1,Output must be outside input directory,True,PASS
4,Q10_optimized_python_rejected,1,Run without Python optimization,True,PASS



=== FINAL QA SUMMARY ===
Path scope fix: VERIFIED
Script SHA-256: 8abc4bbdcec57113fd2e6ed667e6dcf123900b64b3a6ddfd7357f72af9dc5a2b
Full CLI execution: PASS
CSV reproduction: 26/26
JSON reproduction: 1/1
Byte-identical outputs: 27/27
Source metric values: 580/580
Maximum source metric error: 0.000000000000
Negative tests: 5/5
Original sources unchanged: 34/34
Final QA: 10 / 10
Model training: NO
Test evaluation: NO
QA output: /kaggle/working/S9_5/qa
Cell 8 execution time: 3.92 seconds

S9.5 CELL 8: PASS


In [13]:

# ==========================================
# S9.5 - CELL 9: FINAL PACKAGING
# ==========================================

import sys
import os
import json
import shutil
import hashlib
import zipfile
import tempfile
import platform
import time

from pathlib import Path
from datetime import datetime, timezone

pack_start = time.perf_counter()

ZIP_PATH = OUT.parent / "S9_5_QA_Evidence.zip"

LOCKED_SCRIPT_HASH = (
    "8abc4bbdcec57113fd2e6ed667e6dcf123900b64b3a6ddfd7357f72af9dc5a2b"
)

packaging_checks = {}


def packaging_pass(name):

    packaging_checks[name] = "PASS"

    print(
        f"[PASS] {name}",
        flush=True
    )


def write_json(path, data):

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    path.write_text(
        json.dumps(
            data,
            indent=2,
            ensure_ascii=False
        ) + "\n",
        encoding="utf-8"
    )


# ==========================================
# 1. VERIFY FINAL QA AND LOCK SCRIPT
# ==========================================

print("=== FINAL QA VERIFICATION ===", flush=True)

SCRIPT_PATH = OUT / "c_aggregate_results.py"

BUILD_REPORT_PATH = (
    OUT / "aggregation_build_report.json"
)

QA_REPORT_PATH = (
    OUT / "qa" / "aggregation_qa_report.json"
)

assert SCRIPT_PATH.is_file()
assert BUILD_REPORT_PATH.is_file()
assert QA_REPORT_PATH.is_file()

assert sha256_file(SCRIPT_PATH) == (
    LOCKED_SCRIPT_HASH
)

build_disk = json.loads(
    BUILD_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)

qa_disk = json.loads(
    QA_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)

assert build_disk["script_sha256"] == (
    LOCKED_SCRIPT_HASH
)

assert qa_disk["script_sha256"] == (
    LOCKED_SCRIPT_HASH
)

assert qa_disk["build_report_sha256"] == (
    sha256_file(BUILD_REPORT_PATH)
)

assert build_disk["total_passed"] == 8

assert qa_disk["status"] == "AGGREGATION_QA_PASS"

assert qa_disk["final_checks_total"] == 10

assert qa_disk["final_checks_passed"] == 10

assert len(qa_disk["final_checks"]) == 10

assert all(
    status == "PASS"
    for status in qa_disk["final_checks"].values()
)

assert qa_disk["full_cli_execution_verified"]

assert qa_disk["path_scope_fix_verified"]

assert qa_disk["byte_identical_files"] == 27

assert qa_disk["source_metric_values_verified"] == 580

assert qa_disk["negative_tests_passed"] == 5

assert qa_disk["original_sources_unchanged"]

assert not qa_disk["model_training_performed"]

assert not qa_disk["test_evaluation_performed"]

assert not qa_disk["final_evaluation_performed"]

packaging_pass("P01_final_QA_and_script_lock")


# ==========================================
# 2. VERIFY REPRODUCED RESULT FILES
# ==========================================

print("\n=== RESULT FILE INTEGRITY ===")

RESULT_FILES = sorted(
    list(EXPECTED_TABLES)
    + ["sensitivity_checks.json"]
)

assert len(RESULT_FILES) == 27

reproduction_disk = pd.read_csv(
    OUT / "qa" / "reproduction_audit.csv"
)

assert len(reproduction_disk) == 27

assert reproduction_disk["filename"].is_unique

assert set(
    reproduction_disk["filename"]
) == set(RESULT_FILES)

assert (
    reproduction_disk["status"] == "PASS"
).all()

assert (
    reproduction_disk["identical_bytes"] == True
).all()

reproduction_map = reproduction_disk.set_index(
    "filename"
)

for filename in RESULT_FILES:

    path = OUT / filename

    assert path.is_file(), filename

    actual_hash = sha256_file(path)

    expected = reproduction_map.loc[
        filename
    ]

    assert actual_hash == (
        expected["notebook_sha256"]
    ), filename

    assert actual_hash == (
        expected["cli_sha256"]
    ), filename

    if filename.endswith(".csv"):

        assert len(
            pd.read_csv(path)
        ) == EXPECTED_TABLES[filename]

    print(f"[VERIFIED] {filename}")

packaging_pass("P02_result_integrity_27_of_27")


# ==========================================
# 3. VERIFY QA EVIDENCE
# ==========================================

print("\n=== QA EVIDENCE ===")

QA_FILES = [

    "aggregation_qa_report.json",

    "reproduction_audit.csv",

    "source_metric_audit.csv",

    "negative_test_audit.csv",

    "full_cli_execution.txt",

    "Q06_corrupted_source_rejected.txt",

    "Q07_corrupted_registry_rejected.txt",

    "Q08_nonempty_output_rejected.txt",

    "Q09_nested_output_rejected.txt",

    "Q10_optimized_python_rejected.txt"
]

assert len(QA_FILES) == 10

for filename in QA_FILES:

    path = OUT / "qa" / filename

    assert path.is_file(), filename

    assert path.stat().st_size > 0

    print(f"[VERIFIED] {filename}")

metric_disk = pd.read_csv(
    OUT / "qa" / "source_metric_audit.csv"
)

negative_disk = pd.read_csv(
    OUT / "qa" / "negative_test_audit.csv"
)

assert len(metric_disk) == 580

assert (
    metric_disk["status"] == "PASS"
).all()

assert (
    metric_disk["absolute_error"] <= 1e-12
).all()

assert len(negative_disk) == 5

assert (
    negative_disk["status"] == "PASS"
).all()

assert (
    negative_disk[
        "invalid_execution_rejected"
    ] == True
).all()

packaging_pass("P03_QA_evidence_10_of_10")


# ==========================================
# 4. VERIFY SOURCE PACKAGE
# ==========================================

print("\n=== ORIGINAL SOURCE INTEGRITY ===")

assert sha256_file(REGISTRY_PATH) == (
    EXPECTED_REGISTRY_HASH
)

assert len(registry) == 34

assert registry["relative_path"].is_unique

for row in registry.itertuples(index=False):

    source_path = (
        SOURCE_ROOT / row.relative_path
    ).resolve()

    assert source_path.is_relative_to(
        SOURCE_ROOT.resolve()
    )

    assert source_path.is_file()

    assert sha256_file(source_path) == (
        row.sha256
    )

    assert source_path.stat().st_size == (
        row.bytes
    )

packaging_pass("P04_original_sources_34_of_34")


# ==========================================
# 5. CREATE ENVIRONMENT AND RUN CONFIG
# ==========================================

environment = {

    "python": platform.python_version(),

    "numpy": np.__version__,

    "pandas": pd.__version__,

    "platform": platform.platform(),

    "execution_environment": "Kaggle Notebook",

    "accelerator_required": False,

    "internet_required": False
}

run_config = {

    "task": "S9.5_result_aggregation",

    "scope": "HISTORICAL_NON_TEST",

    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "seed_reference": 42,

    "primary_metric": "Average Precision",

    "source_registry_sha256":
        EXPECTED_REGISTRY_HASH,

    "script_sha256":
        LOCKED_SCRIPT_HASH,

    "source_owners": ["B", "C"],

    "source_files": 34,

    "source_csv_files": 29,

    "source_csv_rows": 230,

    "model_configurations": 38,

    "cv_records": 34,

    "validation_configurations": 19,

    "validation_records": 32,

    "feature_cv_records": 25,

    "group_cv_comparisons": 3,

    "validation_threshold_rules": [

        "thr_0.5",

        "thr_val_maxF1",

        "thr_val_maxF2"
    ],

    "protocol_reference": {

        "split_ratio": [0.70, 0.15, 0.15],

        "stratified": True,

        "seed": 42,

        "preprocessing_fit_scope":
            "train or CV training fold only",

        "test_based_selection_allowed": False,

        "protocol_compliance_fully_reverified":
            False
    },

    "model_training": False,

    "raw_brfss_used": False,

    "test_metrics_aggregation": False,

    "final_evaluation": False,

    "official_pipeline_selected": False,

    "team_cross_review_completed": False,

    "team_approval_status": "PENDING"
}


# ==========================================
# 6. PREPARE README
# ==========================================

readme = f"""# S9.5 - Historical Result Aggregation

## 1. Purpose

Aggregate existing cross-validation and validation
results from team members B and C.

This task performs no model training, no new model
selection, no holdout-test evaluation and no FINAL
evaluation.

Primary metric: Average Precision (AP).

## 2. Verified input inventory

- Source files: 34
- B source files: 13
- C source files: 21
- CSV source files: 29
- Total historical CSV rows: 230

The input snapshot includes only registry-listed
source files and the original source registry.

Original archive hashes recorded in the registry
are provenance metadata. The complete upstream
branch archives are not reverified by this package.

## 3. Consolidated outputs

- Model configurations: 38
- Main CV records: 34
- C hyperparameter candidates: 80
- Validation configurations: 19
- Validation records: 32
- Feature selection CV records: 25
- Threshold sensitivity comparisons: 13
- Group-vector CV comparisons: 3

Configuration counts are not counts of independent
newly trained models.

## 4. Quality assurance

- Source integrity: 34/34
- Model registry provenance: 38/38
- Confusion matrix metric checks: 288/288
- Sensitivity checks: 12/12
- Script build checks: 8/8
- Final QA: 10/10
- Byte-identical reproduction: 27/27
- Source metric checks: 580/580
- Negative tests: 5/5

The build report records the state before full CLI
execution. The later aggregation QA report
documents successful full CLI reproduction.

## 5. Reproduction instructions

Tested environment:

- Python {environment["python"]}
- NumPy {environment["numpy"]}
- Pandas {environment["pandas"]}

Install required dependencies if necessary:

    python -m pip install -r requirements.txt

From the extracted S9_5 directory, run:

    python c_aggregate_results.py --input-root inputs/S9_5_inputs --out-dir reproduced_results

The output directory must be new or empty.

Do not use Python optimization mode (-O).

The aggregation process does not require raw
BRFSS data, GPU acceleration or Internet access.

Expected output: 26 CSV files and one JSON file.

## 6. File organization

- c_aggregate_results.py: standalone aggregation tool
- results/: consolidated historical results
- qa/: QA reports, audits and execution logs
- inputs/: original verified aggregation snapshot
- aggregation_build_report.json: build provenance
- s9_5_environment.json: environment information
- s9_5_run_config.json: aggregation scope and protocol
- sha256_manifest.json: file integrity manifest

Source row numbers are one-based data-row positions,
excluding the CSV header.

## 7. Limitations

- Reproduction verifies result aggregation, not
  model retraining or independent model performance.

- AP and ROC-AUC were checked against stored results,
  not recalculated from original prediction scores.

- Validation metrics are not holdout-test metrics.

- Historical code/lock version mismatch reported
  for B remains unresolved.

- Official A4/A5 results are not included.

- Matching reported metrics do not establish
  independent evidence or identical predictions.

- Group-vector CV does not establish statistical
  equivalence or eliminate every leakage risk.

- ZIP-input execution mode has not been independently
  tested by the current aggregation QA.

- Cross-platform byte-identical reproduction
  has not been established.

- Full upstream protocol compliance was not
  independently reverified in this task.

- No causal, future-risk or clinical diagnostic
  claims are supported by this aggregation.

## 8. Status

Technical aggregation QA: PASS

Team cross-review: PENDING

Official FINAL pipeline approval: PENDING

FINAL evaluation: NOT PERFORMED IN THIS TASK

## 9. Locked script SHA-256

{LOCKED_SCRIPT_HASH}
"""


# ==========================================
# 7. BUILD CLEAN STAGING DIRECTORY
# ==========================================

print("\n=== BUILD PACKAGE ===", flush=True)

with tempfile.TemporaryDirectory(

    prefix="s9_packaging_",

    dir=str(OUT.parent)

) as temp_dir:

    temp_dir = Path(temp_dir)

    stage = temp_dir / "S9_5"

    results_dir = stage / "results"

    qa_dir = stage / "qa"

    inputs_dir = (
        stage / "inputs" / "S9_5_inputs"
    )

    for directory in [

        results_dir,
        qa_dir,
        inputs_dir

    ]:

        directory.mkdir(
            parents=True,
            exist_ok=True
        )

    # Copy only explicitly approved outputs.

    for filename in RESULT_FILES:

        shutil.copy2(
            OUT / filename,
            results_dir / filename
        )

    # Copy QA evidence.

    for filename in QA_FILES:

        shutil.copy2(
            OUT / "qa" / filename,
            qa_dir / filename
        )

    # Copy executable and build report.

    shutil.copy2(
        SCRIPT_PATH,
        stage / SCRIPT_PATH.name
    )

    shutil.copy2(
        BUILD_REPORT_PATH,
        stage / BUILD_REPORT_PATH.name
    )

    # Preserve the original source registry bytes.

    shutil.copy2(

        REGISTRY_PATH,

        inputs_dir / "source_registry.csv"
    )

    assert sha256_file(
        inputs_dir / "source_registry.csv"
    ) == EXPECTED_REGISTRY_HASH

    # Copy only the 34 registry-listed source files.

    for row in registry.itertuples(index=False):

        source = SOURCE_ROOT / row.relative_path

        destination = (
            inputs_dir / row.relative_path
        ).resolve()

        assert destination.is_relative_to(
            inputs_dir.resolve()
        )

        assert not destination.exists()

        destination.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        shutil.copy2(
            source,
            destination
        )

        assert sha256_file(destination) == (
            row.sha256
        )

    # Write metadata and documentation.

    write_json(
        stage / "s9_5_environment.json",
        environment
    )

    write_json(
        stage / "s9_5_run_config.json",
        run_config
    )

    (stage / "README_S9_5.md").write_text(

        readme,

        encoding="utf-8"
    )

    (stage / "requirements.txt").write_text(

        f"numpy=={np.__version__}\n"
        f"pandas=={pd.__version__}\n",

        encoding="utf-8"
    )

    # Verify copied files.

    assert sha256_file(
        stage / SCRIPT_PATH.name
    ) == LOCKED_SCRIPT_HASH

    for filename in RESULT_FILES:

        assert sha256_file(
            results_dir / filename
        ) == reproduction_map.loc[
            filename,
            "notebook_sha256"
        ]

    for filename in QA_FILES:

        assert sha256_file(
            qa_dir / filename
        ) == sha256_file(
            OUT / "qa" / filename
        )

    assert len([
        p for p in results_dir.rglob("*")
        if p.is_file()
    ]) == 27

    assert len([
        p for p in qa_dir.rglob("*")
        if p.is_file()
    ]) == 10

    assert len([
        p for p in inputs_dir.rglob("*")
        if p.is_file()
    ]) == 35

    staged_files = sorted(

        p for p in stage.rglob("*")

        if p.is_file()
    )

    assert len(staged_files) == 78

    packaging_pass("P05_clean_package_inventory")


    # ======================================
    # 8. GENERATE SHA-256 MANIFEST
    # ======================================

    print("\n=== SHA-256 MANIFEST ===")

    manifest_entries = []

    for path in staged_files:

        relative_path = path.relative_to(
            stage
        ).as_posix()

        manifest_entries.append({

            "path": relative_path,

            "sha256": sha256_file(path),

            "bytes": path.stat().st_size
        })

    manifest = {

        "algorithm": "SHA-256",

        "root_directory": "S9_5",

        "self_included": False,

        "file_count": len(manifest_entries),

        "files": manifest_entries
    }

    manifest_path = stage / "sha256_manifest.json"

    write_json(
        manifest_path,
        manifest
    )

    assert len(manifest_entries) == 78

    # Verify every manifest entry.

    for entry in manifest_entries:

        path = stage / entry["path"]

        assert path.is_file()

        assert sha256_file(path) == (
            entry["sha256"]
        )

        assert path.stat().st_size == (
            entry["bytes"]
        )

    packaging_pass("P06_manifest_78_of_78")


    # ======================================
    # 9. CREATE ZIP ARCHIVE
    # ======================================

    print("\n=== CREATE ZIP ===")

    temporary_zip = (
        temp_dir / "S9_5_QA_Evidence.zip"
    )

    package_files = sorted(

        p for p in stage.rglob("*")

        if p.is_file()
    )

    assert len(package_files) == 79

    with zipfile.ZipFile(

        temporary_zip,

        mode="w",

        compression=zipfile.ZIP_DEFLATED,

        compresslevel=9

    ) as archive:

        for path in package_files:

            archive.write(

                path,

                arcname=path.relative_to(
                    temp_dir
                ).as_posix()
            )

    packaging_pass("P07_zip_created")


    # ======================================
    # 10. VERIFY ZIP CONTENTS
    # ======================================

    print("\n=== ZIP INTEGRITY AUDIT ===")

    with zipfile.ZipFile(
        temporary_zip,
        mode="r"
    ) as archive:

        assert archive.testzip() is None

        members = archive.namelist()

        assert len(members) == 79

        assert len(set(members)) == 79

        expected_members = {

            f"S9_5/{entry['path']}"

            for entry in manifest_entries

        } | {"S9_5/sha256_manifest.json"}

        assert set(members) == (
            expected_members
        )

        verified_count = 0

        for entry in manifest_entries:

            member = (
                f"S9_5/{entry['path']}"
            )

            data = archive.read(member)

            actual_hash = hashlib.sha256(
                data
            ).hexdigest()

            assert actual_hash == (
                entry["sha256"]
            ), member

            assert len(data) == (
                entry["bytes"]
            ), member

            verified_count += 1

        assert verified_count == 78

        assert archive.read(
            "S9_5/sha256_manifest.json"
        ) == manifest_path.read_bytes()

    packaging_pass("P08_zip_integrity_78_of_78")

    # Publish only after all checks pass.

    verified_zip_hash = sha256_file(
        temporary_zip
    )

    os.replace(
        temporary_zip,
        ZIP_PATH
    )

    assert ZIP_PATH.is_file()

    assert sha256_file(ZIP_PATH) == (
        verified_zip_hash
    )

    packaging_pass("P09_final_zip_published")


# ==========================================
# 11. SAVE EXTERNAL PACKAGING REPORT
# ==========================================

assert len(packaging_checks) == 9

assert all(
    status == "PASS"
    for status in packaging_checks.values()
)

zip_hash = sha256_file(ZIP_PATH)

packaging_report = {

    "task": "S9_5_final_packaging",

    "status": "PASS",

    "scope": "HISTORICAL_NON_TEST",

    "script_sha256": LOCKED_SCRIPT_HASH,

    "zip_filename": ZIP_PATH.name,

    "zip_sha256": zip_hash,

    "zip_bytes": ZIP_PATH.stat().st_size,

    "zip_files": 79,

    "manifest_entries": 78,

    "verified_manifest_entries": 78,

    "result_files": 27,

    "qa_files": 10,

    "input_snapshot_files": 35,

    "packaging_checks": packaging_checks,

    "packaging_checks_passed": 9,

    "team_approval_status": "PENDING",

    "final_evaluation_performed": False
}

PACKAGING_REPORT_PATH = (
    OUT.parent / "S9_5_Packaging_Report.json"
)

write_json(

    PACKAGING_REPORT_PATH,

    packaging_report
)

# ZIP checksum stored separately.

CHECKSUM_PATH = (
    OUT.parent / "S9_5_QA_Evidence.sha256"
)

CHECKSUM_PATH.write_text(

    f"{zip_hash}  {ZIP_PATH.name}\n",

    encoding="utf-8"
)


# ==========================================
# 12. FINAL SUMMARY
# ==========================================

print("\n=== S9.5 FINAL SUMMARY ===")

print("Source files verified: 34/34")

print("Model configurations: 38")

print("CV records: 34")

print("Validation records: 32")

print("Confusion matrix metrics: 288/288")

print("Sensitivity QA: 12/12")

print("Build QA: 8/8")

print("Final QA: 10/10")

print("Byte-identical outputs: 27/27")

print("Source metric checks: 580/580")

print("Negative tests: 5/5")

print("Packaging QA: 9/9")

print("ZIP files: 79")

print("Manifest entries: 78")

print("Manifest verification: 78/78")

print("Input snapshot included: YES")

print("Model training: NO")

print("Test evaluation: NO")

print("FINAL evaluation: NO")

print("Team approval: PENDING")

print("\n=== DELIVERABLES ===")

print("ZIP:", ZIP_PATH)

print(

    "ZIP size (MiB):",

    f"{ZIP_PATH.stat().st_size / 1024**2:.4f}"
)

print("ZIP SHA-256:", zip_hash)

print("Packaging report:", PACKAGING_REPORT_PATH)

print("Checksum:", CHECKSUM_PATH)

print(

    "Execution time:",

    f"{time.perf_counter() - pack_start:.2f} seconds"
)

print("\nS9.5 CELL 9: PASS")

print("S9.5 TECHNICAL AGGREGATION: COMPLETE")


=== FINAL QA VERIFICATION ===
[PASS] P01_final_QA_and_script_lock

=== RESULT FILE INTEGRITY ===
[VERIFIED] b_tuning_recorded.csv
[VERIFIED] c_tuning_candidates.csv
[VERIFIED] cv_coverage_pending.csv
[VERIFIED] cv_summary.csv
[VERIFIED] feature_method_overlap.csv
[VERIFIED] feature_selected_config.csv
[VERIFIED] feature_selection_summary.csv
[VERIFIED] feature_validation_summary.csv
[VERIFIED] group_cv_comparison.csv
[VERIFIED] group_cv_summary.csv
[VERIFIED] group_fold_audit.csv
[VERIFIED] group_overlap_summary.csv
[VERIFIED] imbalance_selected_strategy.csv
[VERIFIED] imbalance_summary.csv
[VERIFIED] model_registry.csv
[VERIFIED] permutation_importance.csv
[VERIFIED] selected_feature_registry.csv
[VERIFIED] sensitivity_checks.json
[VERIFIED] source_integrity_audit.csv
[VERIFIED] source_registry.csv
[VERIFIED] threshold_sensitivity_summary.csv
[VERIFIED] validation_default_05.csv
[VERIFIED] validation_main_F2.csv
[VERIFIED] validation_metrics_recomputed.csv
[VERIFIED] validation_overla